# 05 · Model training, evaluation and horizon-scan export

**Data streams in:** the master files written by notebooks 01–04.
**Out:** a trained invasion-risk model (a small ensemble), cross-validation
metrics including limited-data diagnostics, a static checkpoint for fast
re-scoring, and the CSV files the front-end reads (ranked lists, feature
contributions and GNN saliency maps).

---

## What this notebook does

For every *(species, location)* pair where the species has **not** been
recorded yet, the model estimates how likely the species is to establish
as an alien there. It combines seven signals ("streams"):

| Stream | Question it answers | Main inputs |
|---|---|---|
| **A · Relational ecology** | Does the species' place in the interaction and taxonomy network resemble that of known invaders of this location? | GloBI interaction graph + GBIF taxonomy (GNN embedding) × a compressed summary of all location features |
| **B · Climate niche** | Is the target location's climate inside the species' realised niche? | 10 climate variables (incl. extremes), Köppen–Geiger profile overlap |
| **C · Per-axis suitability** | Which individual climate axes match or mismatch? | same as B |
| **D · Trade pathways** | How much traded material flows into the target from where the species lives? | 11 BACI pathway channels × 2 time windows |
| **E · Biotic opportunity** | Are the species' food, hosts and mutualists — or its enemies — already there? | GBIF presence of GloBI partners (incl. genus/family partners) |
| **F · Location context** | Is the target location intrinsically open to invasion (island, wealth, density, native richness…)? | geography + socio-economy from notebook 01 |
| **G · Spread** | Is the species already established nearby, and how large is its known range? | nearest range location, neighbours within 1 000 km, same SInAS region, range size |

### Labels: positive–unlabeled (PU)

*Positives* are SInAS established-alien records. Every other pair is
*unlabeled*: a real absence, an undetected invasion, or a species that is
already present. Negatives are therefore sampled only from pairs **not
recorded anywhere** (SInAS any status, GBIF presence), and every
range-based feature (niche, Köppen profile, trade routing, spread) is
computed **as if the scored pair itself were not recorded** (pair-level
leave-one-out). Otherwise the model learns "target inside the known
range ⇒ positive", which cannot happen at scan time.

## How the model deals with limited and uneven data

The inputs are uneven in three ways:

1. **Documentation differs between species.** Some species have hundreds
   of GloBI interactions and a well-described native range; others have
   none. Documentation partly *follows* invasion: once a species is a
   known invader, it gets studied. A model that reads "well documented"
   as "invasive" ranks the famous invaders high and overlooks poorly
   studied species, which are often the ones a horizon scan should find.
2. **Estimates from few data points are noisy.** A climate niche from two
   locations, or a biotic opportunity from one known partner, can be
   extreme by chance.
3. **Recording differs between locations.** Europe, North America and
   Oceania contribute most SInAS records.

The guiding rule: ***how much* is known about a species should not drive
its score; *what* is known should, weighted by how reliable it is.**

| # | Problem | Where it enters | Control in this notebook | Setting |
|---|---|---|---|---|
| 1 | Documentation read as invasiveness | species node features (GloBI records, partner counts), biotic partner counts, "native range known" | These **effort features** stay inputs during training, so they absorb the documentation signal, and are set to the value of a typical known invader when scoring and explaining. Scores then mean "risk if this species were documented like a typical invader" (bias-covariate approach, Warton et al. 2013). | `EFFORT_CORRECTION` |
| 2 | The same, through the negatives | uniform negatives are mostly poorly documented species, so "obscure ⇒ negative" is an easy rule | Negatives are drawn so that their documentation level and taxonomic group have the same distribution as in the positives (a target-group background, Phillips et al. 2009). Replaces the degree filter of the hard negatives. | `MATCH_NEGATIVES_ON`, `EFFORT_BINS` |
| 3 | Widespread invaders dominate training | a species alien in 80 locations gave 80 positives | Positives are weighted by *n*<sup>−0.5</sup> per species. | `POSITIVE_WEIGHT_POWER` |
| 4 | Well-recorded locations dominate the positives | uniform negative locations make "well-recorded location ⇒ positive" learnable | Half of the negatives are placed in the locations of the positives. | `NEG_LOCATION_MATCHED` |
| 5 | Noisy niche for small ranges | before: 1 location → global SD (lenient); 2–3 locations → SD at the 0.05 floor, so the climate gate closed for almost every target | The niche variance is shrunk toward a typical niche width, consistently for every range size. | `NICHE_PRIOR_STRENGTH` |
| 6 | Noisy biotic opportunity for few partners | 1 known partner → 0 % or 100 % | Shrunk toward the location's average partner presence. | `BIOTIC_PRIOR_STRENGTH` |
| 7 | Partner counts mix diet breadth and study effort | species node features | Interaction *composition* (share of resources, enemies, mutualists, competitors) is added as shrunk features that do not grow with effort. | `COMPOSITION_PRIOR_STRENGTH` |
| 8 | Training rarely shows data-poor versions of species | — | Range thinning (drop up to 75 % of a pair's range locations) and DropEdge (drop interaction edges) simulate sparse documentation during training (Rong et al. 2020). | `RANGE_THINNING_*`, `EDGE_DROPOUT` |
| 9 | Missing values read as real values | Köppen–Geiger zeros for locations without data; missing native range read as 0 locations; locations without trade data | Availability flags and imputation | — |
| 10 | Uncertainty ignored data gaps | MC dropout only | A seed ensemble on top of MC dropout (Lakshminarayanan et al. 2017). Every exported species also gets a `data_support` score and an `effort_shift` (how much the effort correction moved its score). | `ENSEMBLE_SIZE` |

**Kept on purpose.** Range size is *not* neutralised: it carries "already
invasive elsewhere", the strongest single predictor in horizon scanning.
Its recording-effort part cannot be separated from it with these inputs.

**Not fixed here** (see `model_training_metadata.md`): GBIF presence and
trade are not dated; notebook 04 stores GBIF presence but not GBIF record
counts, so GBIF recording effort cannot be used as a covariate; absolute
probabilities still differ between locations with different recording,
so compare species *within* a location.

**Did it work?** Step 11 reports the ranking metric per data-support
class (low / medium / high), and Step 13 shows how strongly the top lists
follow documentation with and without the correction.

**References**

* Warton DI, Renner IW, Ramp D (2013) Model-based control of observer
  bias for the analysis of presence-only data in ecology. *PLoS ONE* 8:
  e79168.
* Phillips SJ, Dudík M, Elith J, et al. (2009) Sample selection bias and
  presence-only distribution models: implications for background and
  pseudo-absence data. *Ecological Applications* 19: 181–197.
* Rong Y, Huang W, Xu T, Huang J (2020) DropEdge: towards deep graph
  convolutional networks on node classification. *ICLR 2020*.
* Lakshminarayanan B, Pritzel A, Blundell C (2017) Simple and scalable
  predictive uncertainty estimation using deep ensembles. *NeurIPS 30*.

## Version history

**Version 3 (this notebook):** the limited-data controls above.
Cross-validation AUCs are now computed against *matched* negatives and
weighted per species, so they are lower than in version 2 and not
directly comparable; `roc_auc_uniform_negatives` is the version-2
definition. Compare versions on the ranking metric and the temporal
hold-out.

**Version 2** replaced
`horizon_scanner_final_GPUsplit_bioticposneg_exporter_strength_221026.ipynb`:

| # | Before | Now |
|---|---|---|
| 1 | `aligned_country_climate.csv`: 7 variables, min-max scaled | `country_features_model.csv` (notebook 01): ≈100 features, z-scored |
| 2 | Coordinates inside the climate niche | coordinates removed; geography in its own stream (G) |
| 3 | 5 hard-coded trade channels, full-copy allocation of sub-national units | `trade_channels.pt` (notebook 02): 11 deflated, population-allocated channels, full and recent windows |
| 4 | Species-level partners only | genus/family partners and partner counts (notebook 04) |
| 5 | A second copy of the GloBI ontology | `gnn_category` from notebook 03 |
| 6 | Taxonomy edges only for SInAS species | `taxonomy_backbone.csv`: every node |
| 7 | `establishmentMeans` re-parsed | `is_alien` / `is_native` from notebook 03 |
| 8 | Duplicate positives could sit in training and validation | positives de-duplicated |
| 9 | Negatives from GloBI-only species (never positive) | one species pool (`SPECIES_POOL`) |
| 10 | GNN layers shared weights | separate modules per layer |
| 11 | Graph-mode LayerNorm | per-node LayerNorm |
| 12 | "Enemy in-degree" counted victims | counts enemies |
| 13 | Gate detach comment wrong | gate *output* detached |
| 14 | MC dropout also switched on the topology-stream drop | only `nn.Dropout` layers |
| 15 | 2-layer GNN never reached class/kingdom | kingdom and main classes as species features |
| 16 | One-directional symmetric relations, few rank pairs | both directions, every rank pair |
| 17 | Duplicate code paths | one implementation each |
| 18 | Unused country edges in the graph | removed |

## Pipeline at a glance

| Step | What happens | Main objects |
|---|---|---|
| 0 | Imports, configuration, helpers | `INPUTS`, settings |
| 1 | Load and check notebook 01–04 outputs | data frames, tensors |
| 2 | Index maps, de-duplicated positives, species pool | `pos_sp_all`, `pool_mask` |
| 3 | Location features, Köppen–Geiger availability, distances | `co_niche`, `co_context` |
| 4 | Trade channels, no-trade flag | `trade_tensor` |
| 5 | Presence, shrunk biotic opportunity, range and exclusion masks | `base_range_mat`, `exclude_mask` |
| 6 | Graph, species features, effort features, data support | `x_dict`, `species_info` |
| 7 | Model architecture (all classes) | `InvasionRiskModel`, `PairFeatureBuilder` |
| 8 | Pre-train the location autoencoder | `co_latent` |
| 9 | Device setup, matched negative sampling, ranking metric | `NegativeSampler` |
| 10 | Train + evaluate one split | `train_and_evaluate_split` |
| 11 | Cross-validation (random, species, temporal) | `cv_results` |
| 12 | Production ensemble | `members` |
| 13 | Diagnostics, incl. limited-data checks | plots, tables |
| 14 | Static checkpoint | `horizon_scan_static.pt` |
| 15 | Scanner: scoring, uncertainty, contributions, GNN saliency | `Scanner` |
| 16 | Export CSVs for the front-end | `model_outputs/exports/` |
| 17 | Re-load the checkpoint in a fresh session | `load_static_scanner` |

## Inputs (all produced by notebooks 01–04, in `../data/`)

| File | From | Used for |
|---|---|---|
| `country_features_model.csv`, `country_features_schema.json` | 01 | niche, context and autoencoder inputs |
| `complete_country_features_master.csv` | 01 | coordinates and SInAS region (spread stream) |
| `country_koppen_fractions.csv` | 01 | Köppen–Geiger profiles |
| `trade_channels.pt` | 02 | trade stream |
| `sinas_matched_species.csv` | 03 | labels, native range, first-record year, habitat |
| `matched_globi_network.csv`, `matched_globi_network_higher_order.csv` | 03 | interaction edges |
| `taxonomy_backbone.csv` | 03 | taxonomy edges, names, groups |
| `gbif_presence.pt` | 04 | presence, biotic opportunity, partner counts |
| `eicat_lookup.csv` *(optional)* | manual | EICAT category per GBIF species key, shown in the front-end |

**Optional EICAT file.** The model does not predict EICAT categories. To
show them in the front-end, place a CSV with columns `gbif_id` and
`eicat` (MC, MN, MO, MR, MV, DD) in `../data/eicat_lookup.csv`. Global
assessments are compiled in the IUCN GISD
(<https://www.iucngisd.org/gisd/>) and the EICAT pages of the IUCN
(<https://www.iucn.org/resources/conservation-tool/environmental-impact-classification-alien-taxa>);
match names to GBIF keys the same way as notebook 03. Species without a
row are shown as NE (not evaluated).

## Outputs (`../data/model_outputs/`)

| File | Content |
|---|---|
| `cv_results.json` | Metrics of every split, limited-data diagnostics |
| `horizon_scan_static.pt` | Everything needed to re-score without the data files |
| `exports/scan_<location>.csv` | Ranked list: one row per species (front-end table), now with `data_support`, `effort_shift` and `uncertainty_ensemble` |
| `exports/contributions_<location>.csv` | One row per species × feature contribution (front-end detail panel) |
| `exports/gnn_saliency_nodes_<location>.csv`, `…_edges_…` | Local GNN saliency map per species |
| `exports/model_card.json` | Data versions, streams, controls, metrics (front-end "About" tab) |

The file names and all columns of version 2 are unchanged, so the
front-end reads the new files as before. Column definitions are in
`model_training_metadata.md`.

## How to run

1. `pip install torch torch_geometric scikit-learn pandas matplotlib tqdm`
   (`pygbif` only if `GBIF_VERIFY = True`).
2. Run notebooks 01 → 02 → 03 → 04 first.
3. **First a quick test:** set `QUICK_RUN = True` at the end of the Step 0
   cell and *Run All*. It uses tiny settings (minutes, results not
   meaningful) to check that every step runs on your data. Then set it
   back to `False`.
4. *Run All*. With defaults this trains 11 cross-validation models and
   `ENSEMBLE_SIZE` = 3 production models; with a GPU roughly 1.5–4 h. For
   a production-only run set `RUN_RANDOM_CV`, `RUN_SPECIES_CV` and
   `RUN_TEMPORAL_CV` to `False`.
5. To re-score later without training, run Step 0, Step 7, Step 15 and
   Step 17 only.

In [ ]:
import copy
import json
import math
import os
import re
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (auc, precision_recall_curve, roc_auc_score,
                             roc_curve)
from torch_geometric.nn import HeteroConv, SAGEConv
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 60)

## Step 0 · Configuration

Every path and setting lives in the next cell, with a comment on what it
does. Normally you only change `DATA`, the evaluation switches,
`ENSEMBLE_SIZE` and `EXPORT_LOCATIONS`.

**Settings that change the model** (retrain after changing them):
`SPECIES_POOL`, `NICHE_VARIABLES`, `TRADE_WINDOWS`, `SRC_STRENGTH_AGGS`,
the graph settings, the limited-data controls and the model sizes. A
checkpoint stores the settings it was trained with.

**Limited-data controls.** The defaults are recommended. To reproduce
version 2 for a comparison, set `EFFORT_CORRECTION = "none"`,
`MATCH_NEGATIVES_ON = ()`, `NEG_LOCATION_MATCHED = 0`,
`POSITIVE_WEIGHT_POWER = 0`, `RANGE_THINNING_PROB = 0`,
`EDGE_DROPOUT = 0`, `BIOTIC_PRIOR_STRENGTH = 0`, `ENSEMBLE_SIZE = 1`. The
niche shrinkage cannot be switched off (it fixes a defect), but a small
`NICHE_PRIOR_STRENGTH` such as 0.1 comes close.

In [ ]:
# ------------------------------------------------------------- paths
# Set the environment variable IAS_DATA_DIR to use another data folder.
DATA = Path(os.environ.get("IAS_DATA_DIR", Path.cwd().parent / "data"))
OUT_DIR = DATA / "model_outputs"
EXPORT_DIR = OUT_DIR / "exports"
STATIC_CKPT = OUT_DIR / "horizon_scan_static.pt"

INPUTS = {
    # notebook 01 - location features
    "country_features": DATA / "country_features_model.csv",
    "country_schema": DATA / "country_features_schema.json",
    "country_master": DATA / "complete_country_features_master.csv",
    "koppen": DATA / "country_koppen_fractions.csv",
    # notebook 02 - trade
    "trade": DATA / "trade_channels.pt",
    # notebook 03 - species, interactions, taxonomy
    "sinas_species": DATA / "sinas_matched_species.csv",
    "network": DATA / "matched_globi_network.csv",
    "network_hoi": DATA / "matched_globi_network_higher_order.csv",
    "taxonomy": DATA / "taxonomy_backbone.csv",
    # notebook 04 - presence and biotic opportunity
    "gbif": DATA / "gbif_presence.pt",
}
EICAT_CSV = DATA / "eicat_lookup.csv"  # optional, see header

SEED = 42

# ------------------------------------------------ species pool (Step 2)
# Species that can be a training negative and a scan candidate.
#   "sinas_alien"     : >= 1 established-alien record in SInAS
#                       (recommended)
#   "all_with_signal" : every species with any data
SPECIES_POOL = "sinas_alien"

# ------------------------------------------------ location features (3)
# One variable per physiological axis; the quantile columns describe the
# extremes inside a location. Names are looked up with and without the
# "log1p_" prefix that notebook 01 adds to skewed variables.
NICHE_VARIABLES = [
    "wc_bio01_annual_mean_temp_C",
    "wc_bio02_mean_diurnal_range_C",
    "wc_bio04_temp_seasonality_sd_C",
    "wc_bio05_max_temp_warmest_month_C_p95",
    "wc_bio06_min_temp_coldest_month_C_p05",
    "wc_bio12_annual_precipitation_mm",
    "wc_bio14_precip_driest_month_mm",
    "wc_bio15_precip_seasonality_cv",
    "wc_annual_frost_days_est",
    "wc_aridity_index_p_pet",
]
NICHE_STD_FLOOR = 0.05  # minimum niche SD (z units), avoids huge z-scores
# Geography / socio-economy columns that do NOT enter the context stream.
CONTEXT_EXCLUDE = ["latitude", "longitude"]
PROXIMITY_SCALE_KM = 1000.0  # nearest-range closeness = exp(-d / scale)
NEIGHBOUR_RADIUS_KM = 1000.0  # range locations counted as "nearby"

# ------------------------------------------------ trade (Step 4)
TRADE_WINDOWS = ("full", "recent")  # 1995-2024 and 2015-2024
SRC_STRENGTH_AGGS = ("mean", "max")  # exporter strength of range

# ------------------------------------------------ graph (Step 6)
MIN_EDGE_RECORDS = 1  # GloBI records needed to keep an edge
MIN_EDGES_PER_TYPE = 20  # rarer higher-order edge types are dropped
USE_HIGHER_ORDER_EDGES = True
N_TOP_CLASSES = 12  # most common classes get a one-hot species feature
USE_SINAS_HABITAT = True

# ------------------------------- limited-data controls (header table)
# 1. Documentation effort (GloBI records and partner counts, native
#    range known):
#    "neutralize": inputs during training, set to the value of a typical
#                  known invader for scoring and explanations
#                  (recommended)
#    "drop":       constant in training and scoring (not used at all)
#    "none":       used as they are (version 2)
EFFORT_CORRECTION = "neutralize"
# 2. Negatives are drawn so that these species properties have the same
#    distribution as in the positives: "effort" (documentation level in
#    EFFORT_BINS bins) and/or "group" (front-end taxonomic group).
#    () = species drawn uniformly from the pool (version 2).
MATCH_NEGATIVES_ON = ("group", "effort")
EFFORT_BINS = 5  # bin 0 = no GloBI documentation, then quantile bins
# 3. Positive weight = (alien locations of the species) ** -power.
#    0 = every positive counts once (version 2).
POSITIVE_WEIGHT_POWER = 0.5
# 4. Share of negatives placed in the locations of the positives
#    (weighted like the positives); the rest is spread uniformly.
NEG_LOCATION_MATCHED = 0.5
# 5-7. Shrinkage toward a prior, in pseudo-observations.
NICHE_PRIOR_STRENGTH = 3.0  # niche SD toward a typical niche width (> 0)
NICHE_PRIOR_MIN_LOCATIONS = 10  # ranges used to estimate that width
BIOTIC_PRIOR_STRENGTH = 5.0  # partner share toward the location mean
COMPOSITION_PRIOR_STRENGTH = 3.0  # interaction shares toward pool mean
# 8. Training-only augmentation that simulates sparse documentation.
RANGE_THINNING_PROB = 0.5  # share of training pairs with a thinned range
RANGE_THINNING_KEEP_MIN = 0.25  # they keep 25-100 % of their locations
EDGE_DROPOUT = 0.2  # share of interaction edges dropped per epoch
# 10. Data-support score in the exports (0-1, see Step 6b).
SUPPORT_SATURATION = 20  # range locations / partners that count as full
SUPPORT_CLASS_CUTS = (0.4, 0.7)  # low < 0.4 <= medium < 0.7 <= high

# ------------------------------------------------ model sizes (Step 7)
EMB_DIM = 64  # GNN embedding size
CO_LATENT_DIM = 8  # location autoencoder latent size
TRADE_LATENT_DIM = 16
TOPO_STREAM_DROPOUT = 0.25  # random drop of the whole topology stream

# ------------------------------------------------ training (Steps 8-12)
AE_MAX_EPOCHS = 1000
AE_PATIENCE = 50
AE_VAL_FRACTION = 0.15
N_EPOCHS = 500
EARLY_STOP_PATIENCE = 25
TRAIN_NEG_RATIO = 5  # unlabeled pairs per positive (half "hard")
VAL_NEG_RATIO = 10
NEG_REFRESH_EVERY = 1  # resample negatives every n epochs (speed knob)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 3e-4
LABEL_SMOOTHING = 0.05
GATE_SCORE_WEIGHT = 0.5  # checkpoint score = AUC + w * gate separation
FEATURE_CHUNK = 4096  # pairs per feature-building chunk (memory knob)
ENSEMBLE_SIZE = 3  # production models with different seeds (Step 12)

# ------------------------------------------------ evaluation (11, 13)
N_FOLDS = 5
RUN_RANDOM_CV = True
RUN_SPECIES_CV = True
RUN_TEMPORAL_CV = True
TEMPORAL_CUTOFF_YEAR = 2000
MIN_TEMPORAL_POSITIVES = 200
RANK_EVAL_CANDIDATES = 5000
RANK_EVAL_MAX_LOCATIONS = 60
PRODUCTION_VAL_FRACTION = 0.10
BIAS_DIAG_LOCATIONS = 20  # locations scored for the limited-data check
BIAS_DIAG_TOP_K = 50  # list length inspected per location

# ------------------------------------------------ export (Step 16)
EXPORT_LOCATIONS = ["Belgium"]  # SInAS location names, or "all"
EXPORT_TOP_N = 500  # species per location in scan_<location>.csv
EXPLAIN_TOP_N = 100  # of these, how many get contributions
SALIENCY_TOP_N = 25  # of these, how many get a GNN saliency map
SALIENCY_TOP_K = 15  # graph nodes kept per saliency map
N_MC_SAMPLES = 30  # MC-dropout passes per ensemble member
N_IG_STEPS = 50
GBIF_VERIFY = False  # drop candidates with live GBIF records (slow)
GBIF_COUNTRY_CODES = {"Belgium": "BE"}  # ISO-2 per location to verify
GBIF_SLEEP_S = 0.2

# ------------------------------------------------ quick test run
# True = tiny settings that check in a few minutes that every step runs
# on your data. The results are NOT meaningful. Use False for real runs.
QUICK_RUN = False
if QUICK_RUN:
    N_EPOCHS, EARLY_STOP_PATIENCE, AE_MAX_EPOCHS = 3, 2, 20
    N_FOLDS, ENSEMBLE_SIZE = 2, 2
    RUN_SPECIES_CV, RUN_TEMPORAL_CV = True, False
    EXPORT_TOP_N, EXPLAIN_TOP_N, SALIENCY_TOP_N = 50, 3, 2
    N_MC_SAMPLES, N_IG_STEPS = 4, 8
    BIAS_DIAG_LOCATIONS, RANK_EVAL_MAX_LOCATIONS = 3, 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data folder: {DATA}\nDevice: {DEVICE}"
      + ("\nQUICK_RUN: results are not meaningful" if QUICK_RUN else ""))

### Helper functions

Small utilities used throughout, and a check of the settings. Nothing is
computed yet.

In [ ]:
def set_seed(seed=SEED):
    """Seed numpy and torch (CPU and GPU) for reproducible runs."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def check(condition, message):
    """Raise a ValueError with ``message`` when ``condition`` is false."""
    if not condition:
        raise ValueError(message)


def to_bool(series):
    """Convert a column of True/False, 1/0 or yes/no values to bool."""
    if series.dtype == bool:
        return series
    return (series.astype(str).str.strip().str.lower()
            .isin({"true", "1", "yes", "t", "1.0"}))


def resolve_column(name, columns):
    """Return ``name`` or ``log1p_<name>`` if present in ``columns``."""
    for candidate in (name, f"log1p_{name}"):
        if candidate in columns:
            return candidate
    return None


def slugify(text):
    """Lower-case, underscore-separated version of ``text``."""
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(text)).strip("_").lower()


def pretty_feature(column):
    """Readable label for a feature column (used in the front-end)."""
    label = re.sub(r"^log1p_", "", column)
    label = re.sub(r"^wc_bio\d\d_", "", label)
    label = re.sub(r"^wc_", "", label)
    return label.replace("_", " ")


def haversine_matrix_km(lat_deg, lon_deg):
    """Great-circle distances (km) between all pairs of points."""
    lat = np.radians(np.asarray(lat_deg, dtype=float))
    lon = np.radians(np.asarray(lon_deg, dtype=float))
    dlat = lat[:, None] - lat[None, :]
    dlon = lon[:, None] - lon[None, :]
    a = (np.sin(dlat / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :]
         * np.sin(dlon / 2) ** 2)
    return 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def minmax_columns(t):
    """Scale each column of a 2-D tensor to [0, 1] (constant -> 0)."""
    lo = t.min(dim=0, keepdim=True).values
    hi = t.max(dim=0, keepdim=True).values
    return (t - lo) / (hi - lo).clamp(min=1e-12)


def zscore_frame(df):
    """Standardise every column; constant columns get SD 1."""
    mean = df.mean()
    sd = df.std(ddof=0).replace(0, 1.0)
    return (df - mean) / sd, mean, sd


def positive_int_ids(series):
    """Cast an id column to Int64, treating values <= 0 as missing."""
    values = pd.to_numeric(series, errors="coerce")
    return values.where(values > 0).astype("Int64")


def rank_correlation(a, b):
    """Spearman rank correlation of two 1-D arrays (NaN if constant)."""
    ra = pd.Series(np.asarray(a, dtype=float)).rank().to_numpy()
    rb = pd.Series(np.asarray(b, dtype=float)).rank().to_numpy()
    if ra.std() == 0 or rb.std() == 0:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])


def inverse_cdf_sample(prob, n):
    """``n`` indices drawn with replacement from probabilities ``prob``.

    Works for any size (``torch.multinomial`` limits the number of draws).
    """
    cdf = torch.cumsum(prob.double(), dim=0)
    u = torch.rand(n, device=prob.device, dtype=torch.float64) * cdf[-1]
    return torch.searchsorted(cdf, u, right=True).clamp(
        max=len(prob) - 1)


# --- check the settings once -------------------------------------------
check(EFFORT_CORRECTION in ("neutralize", "drop", "none"),
      f"Unknown EFFORT_CORRECTION: {EFFORT_CORRECTION}")
check(set(MATCH_NEGATIVES_ON) <= {"group", "effort"},
      "MATCH_NEGATIVES_ON may only hold 'group' and 'effort'")
check(EFFORT_BINS >= 2, "EFFORT_BINS must be >= 2")
check(NICHE_PRIOR_STRENGTH > 0, "NICHE_PRIOR_STRENGTH must be > 0")
check(min(BIOTIC_PRIOR_STRENGTH, COMPOSITION_PRIOR_STRENGTH) >= 0,
      "prior strengths must be >= 0")
check(0 <= NEG_LOCATION_MATCHED <= 1, "NEG_LOCATION_MATCHED: 0-1")
check(0 <= RANGE_THINNING_PROB <= 1 and 0 < RANGE_THINNING_KEEP_MIN <= 1,
      "RANGE_THINNING_PROB: 0-1, RANGE_THINNING_KEEP_MIN: (0, 1]")
check(0 <= EDGE_DROPOUT < 1, "EDGE_DROPOUT: 0-1")
check(ENSEMBLE_SIZE >= 1, "ENSEMBLE_SIZE must be >= 1")

## Step 1 · Load and check the master input files

**Why:** every later step assumes the four data streams agree on species
keys (`gbif_id`, GBIF Backbone species key) and location names
(`location`, SInAS raw name). This step stops early, with a message that
says which notebook to re-run, if a file is missing or out of date.

**What:**
1. Check that every input exists.
2. Read the tables and the two tensor files.
3. Check required columns and keys.
4. Light cleaning: types, booleans, first-record year from SInAS
   `eventDate`, minimum evidence per GloBI edge.
5. Note the species whose GBIF fetch failed in notebook 04: for them
   "absent in GBIF" means "unknown", which is shown in the exports
   (`gbif_fetch_failed`).

In [ ]:
print("Step 1: loading the master input files")
missing_inputs = [f"{k}: {p}" for k, p in INPUTS.items() if not p.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing inputs (run notebooks 01-04 first):\n  "
        + "\n  ".join(missing_inputs))

df_sinas = pd.read_csv(INPUTS["sinas_species"], low_memory=False)
df_net = pd.read_csv(INPUTS["network"], low_memory=False)
df_hoi = pd.read_csv(INPUTS["network_hoi"], low_memory=False)
df_tax = pd.read_csv(INPUTS["taxonomy"], low_memory=False)
df_cf = pd.read_csv(INPUTS["country_features"])
df_master = pd.read_csv(INPUTS["country_master"], low_memory=False)
df_kg = pd.read_csv(INPUTS["koppen"])
with open(INPUTS["country_schema"]) as fh:
    cf_schema = json.load(fh)
gbif_data = torch.load(INPUTS["gbif"], map_location="cpu",
                       weights_only=False)
trade_data = torch.load(INPUTS["trade"], map_location="cpu",
                        weights_only=False)

REQUIRED_COLUMNS = {
    "sinas_species": (df_sinas, ["gbif_id", "location", "is_alien",
                                 "is_native"]),
    "network": (df_net, ["source_gbif_id", "target_gbif_id",
                         "gnn_category"]),
    "network_hoi": (df_hoi, ["source_gbif_id", "target_gbif_id",
                             "source_rank", "target_rank",
                             "gnn_category"]),
    "taxonomy": (df_tax, ["gbif_id", "rank"]),
    "country_features": (df_cf, ["location"]),
    "country_master": (df_master, ["location", "latitude", "longitude"]),
    "koppen": (df_kg, ["location"]),
}
for name, (frame, cols) in REQUIRED_COLUMNS.items():
    lacking = [c for c in cols if c not in frame.columns]
    check(not lacking, f"{name} lacks columns {lacking}; re-run the "
                       "notebook that writes it.")
for key in ("presence", "pos_opp_mat", "neg_opp_mat", "species_order",
            "countries_order"):
    check(key in gbif_data, f"gbif_presence.pt lacks '{key}'; re-run 04.")
for key in ("channels_order", "locations_order"):
    check(key in trade_data, f"trade_channels.pt lacks '{key}'; re-run 02.")

# --- SInAS species table ----------------------------------------------
df_sinas = df_sinas.dropna(subset=["gbif_id", "location"]).copy()
df_sinas["gbif_id"] = df_sinas["gbif_id"].astype(np.int64)
df_sinas["location"] = df_sinas["location"].astype(str).str.strip()
df_sinas["is_alien"] = to_bool(df_sinas["is_alien"])
df_sinas["is_native"] = to_bool(df_sinas["is_native"]) & ~df_sinas["is_alien"]
if "eventDate" in df_sinas.columns:
    # SInAS eventDate = year of the first record of the alien population.
    df_sinas["first_year"] = pd.to_numeric(df_sinas["eventDate"],
                                           errors="coerce")
else:
    df_sinas["first_year"] = np.nan


# --- GloBI networks ----------------------------------------------------
def clean_network(frame):
    """Integer ids, lower-case model category, evidence count."""
    frame = frame.dropna(subset=["source_gbif_id", "target_gbif_id"]).copy()
    for col in ("source_gbif_id", "target_gbif_id"):
        frame[col] = frame[col].astype(np.int64)
    frame["gnn_category"] = (frame["gnn_category"].fillna("none")
                             .astype(str).str.lower())
    if "n_records" not in frame.columns:
        frame["n_records"] = 1
    frame["n_records"] = pd.to_numeric(frame["n_records"],
                                       errors="coerce").fillna(1)
    return frame


df_net = clean_network(df_net)
df_hoi = clean_network(df_hoi)
for side in ("source_rank", "target_rank"):
    df_hoi[side] = df_hoi[side].astype(str).str.upper()

# --- species whose GBIF fetch failed (notebook 04) ---------------------
gbif_failed_ids = {int(k) for k in gbif_data.get("permanently_failed", [])}

print(f"   SInAS rows: {len(df_sinas):,} | species-level edges: "
      f"{len(df_net):,} | higher-order edges: {len(df_hoi):,}")
print(f"   taxonomy rows: {len(df_tax):,} | location feature table: "
      f"{df_cf.shape[0]} x {df_cf.shape[1] - 2}")
print(f"   dated alien records: "
      f"{df_sinas.loc[df_sinas['is_alien'], 'first_year'].notna().mean():.0%}")
if gbif_failed_ids or not gbif_data.get("complete", True):
    warnings.warn(f"gbif_presence.pt is incomplete: {len(gbif_failed_ids)} "
                  "species failed the GBIF fetch (flagged in the exports).")

## Step 2 · Index maps, positives and the species pool

**Index maps.** Species and locations get a row number in the order of
`gbif_presence.pt` (notebook 04), so every matrix lines up. The step
checks that this order covers every species of notebook 03; if not,
notebook 04 is older than 03 and must be re-run.

**Positives.** One row per *(species, location)* with an established-alien
record. Several SInAS rows can map to the same pair (synonyms rolled up
in notebook 03, several source datasets), so pairs are **de-duplicated**
and the earliest first-record year is kept.

**Species pool.** The species that may be sampled as negatives and that
appear in scan lists (`SPECIES_POOL`). With `"sinas_alien"` these are the
species that are alien somewhere, which is how horizon scans are usually
framed. GloBI-only species stay in the graph as neighbours; they only
leave the candidate lists.

In [ ]:
print("Step 2: index maps, positives and the species pool")
all_species_ids = [int(s) for s in gbif_data["species_order"]]
all_country_names = [str(c).strip() for c in gbif_data["countries_order"]]
sp_to_idx = {sp: i for i, sp in enumerate(all_species_ids)}
co_to_idx = {co: i for i, co in enumerate(all_country_names)}
num_species, num_countries = len(all_species_ids), len(all_country_names)

expected_species = (set(df_sinas["gbif_id"])
                    | set(df_net["source_gbif_id"])
                    | set(df_net["target_gbif_id"]))
not_in_gbif = expected_species - set(sp_to_idx)
check(not not_in_gbif,
      f"{len(not_in_gbif)} species of notebook 03 are missing from "
      "gbif_presence.pt: re-run notebook 04 after 03.")

unknown_locations = sorted(set(df_sinas["location"]) - set(co_to_idx))
if unknown_locations:
    warnings.warn(f"{len(unknown_locations)} SInAS locations are not in "
                  f"gbif_presence.pt and are dropped: "
                  f"{unknown_locations[:10]}")
    df_sinas = df_sinas[df_sinas["location"].isin(co_to_idx)].copy()
df_sinas["sp_idx"] = df_sinas["gbif_id"].map(sp_to_idx).astype(np.int64)
df_sinas["co_idx"] = df_sinas["location"].map(co_to_idx).astype(np.int64)

# --- positives: one row per (species, location) ------------------------
alien_rows = df_sinas[df_sinas["is_alien"]]
pos_table = (alien_rows.groupby(["sp_idx", "co_idx"], as_index=False)
             .agg(first_year=("first_year", "min")))
n_pos = len(pos_table)
pos_sp_all = torch.as_tensor(pos_table["sp_idx"].values, dtype=torch.long)
pos_co_all = torch.as_tensor(pos_table["co_idx"].values, dtype=torch.long)
pos_year = pos_table["first_year"].to_numpy(dtype=float)
print(f"   species: {num_species:,} | locations: {num_countries} | "
      f"alien rows: {len(alien_rows):,} -> unique positive pairs: "
      f"{n_pos:,} ({len(alien_rows) - n_pos:,} duplicates removed)")

native_table = (df_sinas.loc[df_sinas["is_native"], ["sp_idx", "co_idx"]]
                .drop_duplicates())
n_alien_locations = np.bincount(pos_table["sp_idx"], minlength=num_species)
n_native_locations = np.bincount(native_table["sp_idx"],
                                 minlength=num_species)

# --- species pool --------------------------------------------------------
check(SPECIES_POOL in ("sinas_alien", "all_with_signal"),
      f"Unknown SPECIES_POOL: {SPECIES_POOL}")
pool_mask = (torch.as_tensor(n_alien_locations > 0)
             if SPECIES_POOL == "sinas_alien"
             else torch.ones(num_species, dtype=torch.bool))
print(f"   species pool '{SPECIES_POOL}': {int(pool_mask.sum()):,} species")
print(f"   positives per species: median "
      f"{np.median(n_alien_locations[n_alien_locations > 0]):.0f}, max "
      f"{n_alien_locations.max()} (down-weighted by POSITIVE_WEIGHT_POWER)")

## Step 3 · Location features

Notebook 01 delivers ≈100 features per location (missing values already
median-imputed there, with a `<column>_missing` flag). They are used in
four different ways:

| Use | Columns | Why separate |
|---|---|---|
| **Climate niche** (streams B, C and the climate gate) | `NICHE_VARIABLES`: 10 variables, one per physiological axis, including the extremes p05 (coldest winters) and p95 (hottest summers) | A Mahalanobis-type distance double-counts correlated variables, so it gets a short, non-redundant list. Coordinates are excluded: they are geography, not climate. |
| **Location context** (stream F) | geography + socio-economy + native richness from the schema groups, with their `_missing` flags | Properties of the target that make it open to invasion regardless of species. |
| **Location autoencoder** (stream A) | **all** columns | A compact summary of the whole location for the GNN topology stream. |
| **Köppen–Geiger profile** (stream B) | `country_koppen_fractions.csv` | Area share of each climate class, used to compare the climate *classes* of a species' range with those of the target. |

All features are **z-scored over all locations**. This uses no label
information, so it cannot leak (the locations are not split between
training and validation; the species–location pairs are).

**Köppen–Geiger availability (new).** A location without Köppen–Geiger
data used to get all-zero fractions, which read as "no climate class in
common with anything". It is now flagged (`co_koppen_available`):
range locations without data are left out of a species' profile, and the
Köppen leaf of a pair carries a "data available" input.

Finally a **distance matrix** (great-circle km between location
centroids) and a **same-region matrix** (SInAS biogeographic region) are
built for the spread stream.

In [ ]:
print("Step 3: location features")
cf = df_cf.copy()
cf["location"] = cf["location"].astype(str).str.strip()
check(cf["location"].is_unique, "country_features_model.csv has "
                                "duplicate locations.")
missing_locs = [c for c in all_country_names if c not in set(cf["location"])]
check(not missing_locs, f"{len(missing_locs)} locations have no feature "
                        f"row: {missing_locs[:10]}. Re-run notebook 01.")
cf = cf.set_index("location").reindex(all_country_names)
feature_cols = [c for c in cf.columns if c != "locationID"]
cf_values = cf[feature_cols].astype(float)
check(not cf_values.isna().any().any(),
      "NaN in country_features_model.csv; notebook 01 should impute.")
if cf_schema.get("n_features") not in (None, len(feature_cols)):
    warnings.warn("Schema n_features differs from the CSV; using the CSV.")
cf_z, cf_mean, cf_sd = zscore_frame(cf_values)

# --- (a) climate niche variables ---------------------------------------
niche_cols = []
for name in NICHE_VARIABLES:
    col = resolve_column(name, feature_cols)
    if col is None:
        warnings.warn(f"Niche variable '{name}' not found; skipped.")
    else:
        niche_cols.append(col)
check(len(niche_cols) >= 3, "Fewer than 3 niche variables found.")
co_niche = torch.tensor(cf_z[niche_cols].values, dtype=torch.float32)

# --- (b) location context ------------------------------------------------
groups = cf_schema.get("feature_groups", {})
ctx_source = (groups.get("geography", [])
              + groups.get("socioeconomic_and_biodiversity", []))
ctx_base = [c for c in ctx_source
            if c in feature_cols and c not in niche_cols
            and c not in CONTEXT_EXCLUDE]
ctx_flags = [f"{c}_missing" for c in ctx_base
             if f"{c}_missing" in feature_cols]
context_cols = list(dict.fromkeys(ctx_base + ctx_flags))
check(context_cols, "No context columns found in the schema groups.")
co_context = torch.tensor(cf_z[context_cols].values, dtype=torch.float32)

# --- (c) autoencoder input: every feature --------------------------------
co_all = torch.tensor(cf_z[feature_cols].values, dtype=torch.float32)

# --- (d) Koppen-Geiger fractions and their availability ------------------
kg_classes = cf_schema.get("koppen_classes") or [
    c[3:] for c in df_kg.columns
    if c.startswith("kg_") and c != "kg_dominant_class"]
kg_cols = [f"kg_{c}" for c in kg_classes if f"kg_{c}" in df_kg.columns]
kg = (df_kg.assign(location=df_kg["location"].astype(str).str.strip())
      .drop_duplicates("location").set_index("location")
      .reindex(all_country_names)[kg_cols].astype(float))
kg_missing = kg.isna().all(axis=1) | (kg.fillna(0).sum(axis=1) == 0)
kg = kg.fillna(0.0)
kg = kg.div(kg.sum(axis=1).replace(0, 1.0), axis=0)
co_koppen = torch.tensor(kg.values, dtype=torch.float32)
co_koppen_available = torch.as_tensor(~kg_missing.to_numpy())
if kg_missing.any():
    warnings.warn(f"{int(kg_missing.sum())} locations without Koppen-Geiger "
                  f"data (flagged): {list(kg.index[kg_missing])[:10]}")

# --- (e) distances and regions for the spread stream ----------------------
geo = (df_master.assign(location=df_master["location"].astype(str)
                        .str.strip())
       .drop_duplicates("location").set_index("location")
       .reindex(all_country_names))


def coordinate(name):
    """Coordinate from the master table, model table as fallback."""
    values = geo[name].astype(float)
    if name in cf_values.columns:
        values = values.fillna(cf_values[name])
    return values


lat, lon = coordinate("latitude"), coordinate("longitude")
check(not (lat.isna().any() or lon.isna().any()),
      "Some locations have no coordinates.")
dist_km = torch.tensor(haversine_matrix_km(lat, lon), dtype=torch.float32)
if "sinas_region" in geo.columns:
    region_codes = pd.factorize(geo["sinas_region"])[0]
else:
    warnings.warn("No sinas_region column; 'same region' share is 0.")
    region_codes = np.full(num_countries, -1)
missing_region = region_codes < 0
region_codes[missing_region] = -1 - np.arange(int(missing_region.sum()))
same_region = torch.as_tensor(region_codes[:, None] == region_codes[None, :])

n_niche, n_co_features = len(niche_cols), len(feature_cols)
print(f"   niche variables: {n_niche} | context: {len(context_cols)} | "
      f"autoencoder input: {n_co_features} | Koppen classes: "
      f"{len(kg_cols)} ({int(kg_missing.sum())} locations without)")

## Step 4 · Trade channels

`trade_channels.pt` (notebook 02) holds, per pathway channel, the mean
annual trade value from every location to every other one, already
deflated and allocated from BACI countries to SInAS locations.

1. Take the windows in `TRADE_WINDOWS`: **full** (1995–2024) for the
   long-term exposure and **recent** (2015–2024) for current pathways.
   Both are `log1p` of the mean annual value (thousand constant USD), so
   they are on the same scale. Channel names become
   `<channel>__<window>`.
2. Re-order rows and columns to the location order of Step 2 and zero
   the diagonal (no self-trade).
3. Divide each channel by its maximum, so all channels lie in [0, 1].
4. Location profiles: **import strength** (sum over exporters) and
   **export strength** (sum over importers), min-max scaled per channel.
   Both reward breadth of trade partners as well as volume.
5. **No-trade flag (new).** A location with no recorded trade at all
   (no BACI code and no sovereign fallback) would look like a location
   that imports nothing. If there are any, a `no_trade_data` flag is
   added to the location context. Domestic trade (mainland → own islands)
   is not in BACI; that limitation stays (see the metadata).

In [ ]:
print("Step 4: trade channels")
base_channels = [str(c) for c in trade_data["channels_order"]]
trade_locs = [str(c).strip() for c in trade_data["locations_order"]]
trade_pos = {name: i for i, name in enumerate(trade_locs)}
missing_trade = [c for c in all_country_names if c not in trade_pos]
check(not missing_trade, f"{len(missing_trade)} locations missing from "
                         f"trade_channels.pt: {missing_trade[:10]}. "
                         "Re-run notebook 02.")
order = torch.as_tensor([trade_pos[c] for c in all_country_names])


def trade_window(window):
    """log1p(mean annual value) tensor (C, N, N) for one time window."""
    key = {"full": "value_full", "recent": "value_recent"}.get(window)
    check(key is not None, f"Unknown trade window: {window}")
    if key in trade_data:
        return torch.log1p(torch.as_tensor(trade_data[key]).float()
                           .clamp(min=0))
    check(window == "full", f"trade_channels.pt has no '{key}'.")
    return torch.as_tensor(trade_data["trade_tensor"]).float()


window_tensors, trade_channel_names = [], []
for window in TRADE_WINDOWS:
    window_tensors.append(trade_window(window))
    trade_channel_names += [f"{ch}__{window}" for ch in base_channels]
trade_tensor = torch.cat(window_tensors, dim=0)
trade_tensor = trade_tensor[:, order][:, :, order].contiguous()
diag = torch.arange(num_countries)
trade_tensor[:, diag, diag] = 0.0
trade_tensor = trade_tensor / trade_tensor.amax(dim=(1, 2), keepdim=True
                                                ).clamp(min=1e-12)
num_channels = trade_tensor.shape[0]

# importer strength: sum over exporters (dim 1); exporter: over importers
co_trade_import = minmax_columns(trade_tensor.sum(dim=1).T.contiguous())
co_trade_export = minmax_columns(trade_tensor.sum(dim=2).T.contiguous())
SRC_DIM = len(SRC_STRENGTH_AGGS) * num_channels

# --- locations without any recorded trade ------------------------------
no_trade = ((trade_tensor.sum(dim=(0, 1)) == 0)
            & (trade_tensor.sum(dim=(0, 2)) == 0))
if no_trade.any():
    names = [all_country_names[i] for i in no_trade.nonzero().view(-1)]
    warnings.warn(f"{len(names)} locations have no trade data at all "
                  f"(flag added to the context stream): {names[:10]}")
    flag = no_trade.float()
    flag = (flag - flag.mean()) / flag.std(unbiased=False).clamp(min=1e-12)
    co_context = torch.cat([co_context, flag[:, None]], dim=1)
    context_cols = context_cols + ["no_trade_data"]
n_context = len(context_cols)
print(f"   channels: {num_channels} ({len(base_channels)} x "
      f"{len(TRADE_WINDOWS)} windows) | tensor: {tuple(trade_tensor.shape)}"
      f" | locations without trade: {int(no_trade.sum())}")

## Step 5 · Presence, biotic opportunity and the range / exclusion masks

From `gbif_presence.pt` (notebook 04):

* `presence[s, c]`: species *s* is recorded in location *c* (GBIF or
  SInAS, any status). It means "recorded", not "native".
* `pos_opp_mat` / `neg_opp_mat`: share of the species' helpful /
  harmful GloBI partners (including genus and family partners) present
  in *c*.
* `n_pos_partners` / `n_neg_partners`: how many partners there are.

**Shrinkage of the biotic opportunity (new).** With one known partner the
share is 0 % or 100 %; with forty it is a stable estimate. The share is
therefore pulled toward the location's average partner share (the mean
over all species with partners), with `BIOTIC_PRIOR_STRENGTH` = *k*
pseudo-partners:

$$\text{opportunity}_{s,c} = \frac{n_s \cdot \text{share}_{s,c} + k \cdot \overline{\text{share}}_c}{n_s + k}$$

A species without known partners gets the location average (not 0), so
"no partners known" no longer reads as "no partners present". The
species-level part of stream E still says how many partners are known
(`has_*`, log counts); these are effort features (Step 6b). The exports
keep the observed, unshrunk share.

Two masks are built:

* **`base_range_mat`** (S × N): locations that count as the species'
  known range = presence ∪ SInAS native ∪ SInAS alien. It feeds the
  niche, Köppen profile, trade routing and spread features. Per training
  split, the held-out positives are removed from a copy.
* **`exclude_mask`** (S × N): pairs that are never negatives and never
  scan candidates = the range plus every other SInAS row (e.g.
  cryptogenic).

In [ ]:
print("Step 5: presence, biotic opportunity and masks")
presence = torch.as_tensor(gbif_data["presence"]).bool()
check(tuple(presence.shape) == (num_species, num_countries),
      "presence matrix has the wrong shape; re-run notebook 04.")


def partner_counts(key):
    """Partner counts per species from notebook 04 (zeros if absent)."""
    if key not in gbif_data:
        warnings.warn(f"gbif_presence.pt has no '{key}'; using zeros.")
        return torch.zeros(num_species)
    return torch.as_tensor(gbif_data[key]).float().reshape(-1)


def shrink_opportunity(raw, n, k):
    """Pull partner shares toward the per-location mean share.

    Returns (shrunk S x N matrix, prior per location N). ``k`` = 0
    returns the raw shares.
    """
    has = n > 0
    prior = (raw[has].mean(dim=0) if bool(has.any())
             else torch.zeros(raw.shape[1]))
    if k <= 0:
        return raw, prior
    shrunk = (n[:, None] * raw + k * prior[None, :]) / (n[:, None] + k)
    return shrunk, prior


n_pos_partners = partner_counts("n_pos_partners")
n_neg_partners = partner_counts("n_neg_partners")
raw_pos = torch.as_tensor(gbif_data["pos_opp_mat"]).float()
raw_neg = torch.as_tensor(gbif_data["neg_opp_mat"]).float()
pos_opp_mat, pos_prior = shrink_opportunity(raw_pos, n_pos_partners,
                                            BIOTIC_PRIOR_STRENGTH)
neg_opp_mat, neg_prior = shrink_opportunity(raw_neg, n_neg_partners,
                                            BIOTIC_PRIOR_STRENGTH)
few = (n_pos_partners > 0) & (n_pos_partners <= 2)
if bool(few.any()):
    change = (pos_opp_mat[few] - raw_pos[few]).abs().mean()
    print(f"   species with 1-2 helpful partners: {int(few.sum()):,}; mean "
          f"change of their opportunity by shrinkage: {change:.3f}")
del raw_pos, raw_neg

log_pos = torch.log1p(n_pos_partners)
log_neg = torch.log1p(n_neg_partners)
# Species-level part of stream E: [has helpful partners, has harmful
# partners, log #helpful (scaled), log #harmful (scaled)]. All four are
# documentation-effort features (neutralised for scoring, Step 9).
sp_bio_static = torch.stack([
    (n_pos_partners > 0).float(),
    (n_neg_partners > 0).float(),
    log_pos / log_pos.max().clamp(min=1.0),
    log_neg / log_neg.max().clamp(min=1.0),
], dim=1)

native_sp = torch.as_tensor(native_table["sp_idx"].values, dtype=torch.long)
native_co = torch.as_tensor(native_table["co_idx"].values, dtype=torch.long)
base_range_mat = presence.float()
base_range_mat[native_sp, native_co] = 1.0
base_range_mat[pos_sp_all, pos_co_all] = 1.0
n_range_locations = base_range_mat.sum(dim=1).numpy()

exclude_mask = base_range_mat.bool()
all_rows = df_sinas[["sp_idx", "co_idx"]].drop_duplicates()
exclude_mask[torch.as_tensor(all_rows["sp_idx"].values),
             torch.as_tensor(all_rows["co_idx"].values)] = True
print(f"   presences: {int(presence.sum()):,} | range pairs: "
      f"{int(base_range_mat.sum()):,} | excluded pairs: "
      f"{int(exclude_mask.sum()):,} "
      f"({exclude_mask.float().mean():.1%} of the grid)")

## Step 6 · Heterogeneous graph

Node types: `species` plus the taxonomy ranks `genus`, `family`,
`order`, `class`, `phylum`, `kingdom`. Every species node of the
universe (SInAS species **and** GloBI partners) is linked to its genus,
and every taxon to its parent, from `taxonomy_backbone.csv`.

Edge types:

| Relation | From → to | Meaning |
|---|---|---|
| `belongs_to_<rank>` / `has_<rank>` | child ↔ parent | taxonomy |
| `resource` | consumer → resource | eats, parasitises, … (`gnn_category` from notebook 03) |
| `enemy` | victim → enemy | explicit enemy records + every resource edge reversed |
| `mutualism`, `competition` | both directions | symmetric relations |
| `hoi_<category>` / `rev_hoi_<category>` | any rank pair | higher-order (genus/family/order/class) interactions; types with fewer than `MIN_EDGES_PER_TYPE` edges are dropped |

Messages flow along an edge from source to target, so a node is informed
by its **in**-neighbours. Location nodes are not part of the graph: the
GNN only describes species.

A species without GloBI interactions still gets messages from its genus
(whose features are the mean of its species), so it borrows information
from its relatives. During training, `EDGE_DROPOUT` removes a random
share of the *interaction* edges each epoch (taxonomy edges are always
kept), so the GNN also learns from sparse neighbourhoods like those of
poorly studied species.

In [ ]:
print("Step 6: heterogeneous graph")
RANKS = ["genus", "family", "order", "class", "phylum", "kingdom"]
LEVELS = ["species"] + RANKS
HOI_RANK_TYPES = {"SPECIES": "species", "GENUS": "genus",
                  "FAMILY": "family", "ORDER": "order", "CLASS": "class"}
TAXONOMY_PREFIXES = ("belongs_to_", "has_")  # never dropped by DropEdge

tax = df_tax.copy()
tax["rank"] = tax["rank"].astype(str).str.upper()
tax["gbif_id"] = positive_int_ids(tax["gbif_id"])
for rank in RANKS:
    col = f"{rank}_id"
    tax[col] = positive_int_ids(tax[col]) if col in tax.columns else pd.NA

# --- species taxonomy (backbone first, SInAS columns as fallback) -------
sp_tax = pd.DataFrame({"gbif_id": all_species_ids})
sp_tax = sp_tax.merge(
    tax[tax["rank"] == "SPECIES"].drop_duplicates("gbif_id"),
    on="gbif_id", how="left")
sinas_first = df_sinas.drop_duplicates("gbif_id").set_index("gbif_id")
for rank in RANKS:
    fallback_col = f"gbif_{rank}_id"
    if fallback_col in sinas_first.columns:
        fallback = positive_int_ids(sinas_first[fallback_col])
        sp_tax[f"{rank}_id"] = sp_tax[f"{rank}_id"].fillna(
            sp_tax["gbif_id"].map(fallback))
print(f"   species without a genus: {int(sp_tax['genus_id'].isna().sum()):,}")

# --- taxon node sets -----------------------------------------------------
tax_nodes = {rank: set(sp_tax[f"{rank}_id"].dropna().astype(np.int64))
             for rank in RANKS}
if USE_HIGHER_ORDER_EDGES:
    for side in ("source", "target"):
        for rank_name, node_type in HOI_RANK_TYPES.items():
            if node_type == "species":
                continue
            ids = df_hoi.loc[df_hoi[f"{side}_rank"] == rank_name,
                             f"{side}_gbif_id"]
            tax_nodes[node_type].update(ids.astype(np.int64))

# --- parent links, walking up the ranks ----------------------------------
tax_links = {}
for child, parent in zip(LEVELS[:-1], LEVELS[1:]):
    pcol = f"{parent}_id"
    if child == "species":
        links = sp_tax[["gbif_id", pcol]].set_axis(["child", "parent"],
                                                   axis=1)
    else:
        rows = tax.loc[tax["rank"] == child.upper(), ["gbif_id", pcol]]
        from_species = sp_tax[[f"{child}_id", pcol]]
        links = pd.concat([rows.set_axis(["child", "parent"], axis=1),
                           from_species.set_axis(["child", "parent"],
                                                 axis=1)])
    links = links.dropna().astype(np.int64).drop_duplicates("child")
    if child != "species":
        links = links[links["child"].isin(tax_nodes[child])]
    tax_nodes[parent].update(links["parent"].tolist())
    tax_links[(child, parent)] = links

node_index = {"species": sp_to_idx}
for rank in RANKS:
    node_index[rank] = {tid: i for i, tid in
                        enumerate(sorted(tax_nodes[rank]))}
num_nodes = {nt: len(ix) for nt, ix in node_index.items()}

# --- edges ---------------------------------------------------------------
edge_index_dict = {}


def add_edges(src_type, rel, dst_type, src, dst, min_count=1):
    """Store unique (src, dst) edges of one type if there are enough."""
    pairs = np.unique(np.stack([np.asarray(src, dtype=np.int64),
                                np.asarray(dst, dtype=np.int64)]), axis=1)
    if pairs.shape[1] >= min_count:
        edge_index_dict[(src_type, rel, dst_type)] = torch.as_tensor(pairs)


for (child, parent), links in tax_links.items():
    c_idx = links["child"].map(node_index[child])
    p_idx = links["parent"].map(node_index[parent])
    ok = c_idx.notna() & p_idx.notna()
    add_edges(child, f"belongs_to_{parent}", parent, c_idx[ok], p_idx[ok])
    add_edges(parent, f"has_{child}", child, p_idx[ok], c_idx[ok])

# species-level interactions
net = df_net[(df_net["n_records"] >= MIN_EDGE_RECORDS)
             & df_net["gnn_category"].isin(
                 ["resource", "enemy", "mutualism", "competition"])].copy()
net["src"] = net["source_gbif_id"].map(sp_to_idx)
net["dst"] = net["target_gbif_id"].map(sp_to_idx)
net = net.dropna(subset=["src", "dst"])
net = net[net["src"] != net["dst"]].astype({"src": np.int64,
                                            "dst": np.int64})
by_cat = {cat: net[net["gnn_category"] == cat] for cat in
          ("resource", "enemy", "mutualism", "competition")}
res = by_cat["resource"]
add_edges("species", "resource", "species", res["src"], res["dst"])
# enemy edges point victim -> enemy: explicit records + reversed resource
add_edges("species", "enemy", "species",
          np.r_[by_cat["enemy"]["src"], res["dst"]],
          np.r_[by_cat["enemy"]["dst"], res["src"]])
for cat in ("mutualism", "competition"):
    sub = by_cat[cat]
    add_edges("species", cat, "species", np.r_[sub["src"], sub["dst"]],
              np.r_[sub["dst"], sub["src"]])

# higher-order interactions (any rank pair with enough edges)
if USE_HIGHER_ORDER_EDGES:
    hoi = df_hoi[(df_hoi["n_records"] >= MIN_EDGE_RECORDS)
                 & df_hoi["gnn_category"].isin(
                     ["resource", "enemy", "mutualism", "competition"])]
    hoi = hoi.assign(src_type=hoi["source_rank"].map(HOI_RANK_TYPES),
                     dst_type=hoi["target_rank"].map(HOI_RANK_TYPES))
    hoi = hoi.dropna(subset=["src_type", "dst_type"])
    for (st, dt, cat), sub in hoi.groupby(["src_type", "dst_type",
                                           "gnn_category"]):
        s_idx = sub["source_gbif_id"].map(node_index[st])
        d_idx = sub["target_gbif_id"].map(node_index[dt])
        ok = s_idx.notna() & d_idx.notna() & ~((st == dt) & (s_idx == d_idx))
        add_edges(st, f"hoi_{cat}", dt, s_idx[ok], d_idx[ok],
                  MIN_EDGES_PER_TYPE)
        add_edges(dt, f"rev_hoi_{cat}", st, d_idx[ok], s_idx[ok],
                  MIN_EDGES_PER_TYPE)

# keep node types that have nodes and take part in at least one edge type
node_types = [nt for nt in LEVELS if num_nodes[nt] > 0 and any(
    nt in (et[0], et[2]) for et in edge_index_dict)]
edge_index_dict = {et: ei for et, ei in edge_index_dict.items()
                   if et[0] in node_types and et[2] in node_types}
edge_types = list(edge_index_dict)
n_interaction_types = sum(not et[1].startswith(TAXONOMY_PREFIXES)
                          for et in edge_types)
print("   nodes: " + ", ".join(f"{nt} {num_nodes[nt]:,}" for nt in node_types))
print(f"   edge types: {len(edge_types)} ({n_interaction_types} interaction"
      f" types) | edges: "
      f"{sum(e.shape[1] for e in edge_index_dict.values()):,}")

### 6b · Species node features, effort features and data support

Each species node starts from a vector of descriptive traits. None of them
uses the label (where a species is alien), so they cannot leak it.

| Feature | Source | Effort feature? |
|---|---|---|
| `log_n_resources`, `log_n_enemies`, `log_n_mutualists`, `log_n_competitors`, `log_total_interactions` | distinct species-level partners per relation | yes |
| `log_globi_records` | GloBI records involving the species | yes |
| `log_n_pos_partners`, `log_n_neg_partners` | helpful / harmful partners incl. genus and family partners (notebook 04) | yes |
| `native_range_known` | the species has ≥ 1 SInAS native record | yes |
| `share_resources`, `share_enemies`, `share_mutualists`, `share_competitors` **(new)** | composition of the partners, shrunk toward the pool average with `COMPOSITION_PRIOR_STRENGTH` pseudo-partners | no |
| `log_n_native_locations` | SInAS native locations; **imputed** with the pool median when the native range is unknown (before: 0) | no |
| `habitat_terrestrial`, `_freshwater`, `_marine`, `_unknown` | SInAS `habitat` | no |
| `kingdom_*`, `class_*` | taxonomy one-hots (`N_TOP_CLASSES` most common classes in the pool) | no |

Counts are `log1p`-transformed and divided by their maximum.

**Effort features** measure *how much* is documented about a species, not
what it is like. They are listed in `EFFORT_FEATURES`; Step 9 sets them to
the value of a typical known invader when scoring (`EFFORT_CORRECTION`).
The partner *counts* mix diet breadth and study effort, which cannot be
separated with GloBI; the composition shares keep the "what kind of
ecology" information without growing with effort.

**Effort score** (for matching negatives): the mean of the scaled
`log_globi_records`, `log_total_interactions`, `log_n_pos_partners` and
`log_n_neg_partners`.

**Data support** (exported per species, 0–1): the mean of three parts —
range (`log(1 + range locations) / log(1 + SUPPORT_SATURATION)`, capped
at 1), interactions (the same for the number of known partners) and
whether the native range is known. Classes: low < 0.4 ≤ medium < 0.7 ≤
high (`SUPPORT_CLASS_CUTS`). It says how much evidence a prediction rests
on; it does not change the score.

The same step builds the **species information table** (names, taxonomy,
front-end group, support) and the **signal mask**: species with no
interactions, no presence and no native record carry no information and
are never scored.

**Front-end groups** follow the mockup: plant, invertebrate, fish, herp,
bird, mammal, fungi (fungi *and* microbes).

In [ ]:
print("Step 6b: species node features")


def distinct_count(frame, by, other, undirected=False):
    """Distinct partners per species for one relation (array of length S)."""
    if frame.empty:
        return np.zeros(num_species)
    pairs = frame[[by, other]].to_numpy()
    if undirected:
        pairs = np.r_[pairs, pairs[:, ::-1]]
    pairs = np.unique(pairs, axis=0)
    return np.bincount(pairs[:, 0], minlength=num_species).astype(float)


n_res = distinct_count(res, "src", "dst")
# a species' enemies: explicit enemy records (victim -> enemy) plus
# consumers that eat it (reversed resource edges)
victim_enemy = pd.concat([
    by_cat["enemy"][["src", "dst"]],
    res[["dst", "src"]].set_axis(["src", "dst"], axis=1)])
n_enemy = distinct_count(victim_enemy, "src", "dst")
n_mut = distinct_count(by_cat["mutualism"], "src", "dst", undirected=True)
n_comp = distinct_count(by_cat["competition"], "src", "dst",
                        undirected=True)
n_total = n_res + n_enemy + n_mut + n_comp

records = np.zeros(num_species)
for side in ("source_gbif_id", "target_gbif_id"):
    idx = df_net[side].map(sp_to_idx).dropna().astype(np.int64)
    np.add.at(records, idx.values, df_net.loc[idx.index, "n_records"].values)

pool_np = pool_mask.numpy()
sp_feat = {}


def add_log_feature(name, values):
    """log1p(values) / max, stored as a species feature."""
    v = np.log1p(np.asarray(values, dtype=float))
    sp_feat[name] = v / max(v.max(), 1e-12)


# --- effort features (documentation) -------------------------------------
add_log_feature("log_n_resources", n_res)
add_log_feature("log_n_enemies", n_enemy)
add_log_feature("log_n_mutualists", n_mut)
add_log_feature("log_n_competitors", n_comp)
add_log_feature("log_total_interactions", n_total)
add_log_feature("log_globi_records", records)
add_log_feature("log_n_pos_partners", n_pos_partners.numpy())
add_log_feature("log_n_neg_partners", n_neg_partners.numpy())
native_known = n_native_locations > 0
sp_feat["native_range_known"] = native_known.astype(float)

# --- composition of the interactions (shrunk shares) ---------------------
k_comp = COMPOSITION_PRIOR_STRENGTH
pool_total = n_total[pool_np].sum()
for name, counts in (("resources", n_res), ("enemies", n_enemy),
                     ("mutualists", n_mut), ("competitors", n_comp)):
    prior = (counts[pool_np].sum() / pool_total if pool_total > 0
             else 0.25)
    denom = n_total + k_comp
    sp_feat[f"share_{name}"] = np.where(
        denom > 0, (counts + k_comp * prior) / np.maximum(denom, 1e-12),
        prior)

# --- native range size, imputed when unknown ------------------------------
log_native = np.log1p(n_native_locations.astype(float))
known_pool = native_known & pool_np
native_fill = (float(np.median(log_native[known_pool]))
               if known_pool.any() else 0.0)
log_native = np.where(native_known, log_native, native_fill)
sp_feat["log_n_native_locations"] = log_native / max(log_native.max(), 1e-12)

# --- habitat (SInAS), species level: any record with the keyword ----------
habitat = np.zeros((num_species, 3))
if USE_SINAS_HABITAT and "habitat" in df_sinas.columns:
    text = df_sinas["habitat"].fillna("").astype(str).str.lower()
    patterns = [r"terrestr", r"fresh|limnic|inland", r"marine|brackish"]
    for j, pattern in enumerate(patterns):
        hit = df_sinas.loc[text.str.contains(pattern), "sp_idx"].unique()
        habitat[hit, j] = 1.0
for j, name in enumerate(["terrestrial", "freshwater", "marine"]):
    sp_feat[f"habitat_{name}"] = habitat[:, j]
sp_feat["habitat_unknown"] = (habitat.sum(axis=1) == 0).astype(float)

# --- taxonomy names (for one-hots, display and front-end groups) ----------
for rank in ["kingdom", "phylum", "class", "order", "family"]:
    if rank not in sp_tax.columns:
        sp_tax[rank] = pd.NA
kingdom = sp_tax["kingdom"].fillna("").astype(str)
for k in ["Animalia", "Plantae", "Fungi", "Chromista"]:
    sp_feat[f"kingdom_{k.lower()}"] = (kingdom == k).to_numpy(float)
sp_feat["kingdom_other"] = (~kingdom.isin(
    ["Animalia", "Plantae", "Fungi", "Chromista"])).to_numpy(float)
cls = sp_tax["class"].fillna("").astype(str)
top_classes = (cls[pool_np & (cls != "")].value_counts()
               .head(N_TOP_CLASSES).index.tolist())
for c in top_classes:
    sp_feat[f"class_{slugify(c)}"] = (cls == c).to_numpy(float)
sp_feat["class_other"] = (~cls.isin(top_classes)).to_numpy(float)

sp_feature_names = list(sp_feat)
sp_features = torch.tensor(np.column_stack([sp_feat[n] for n in
                                            sp_feature_names]),
                           dtype=torch.float32)
n_sp_features = sp_features.shape[1]

# --- effort features and effort score --------------------------------------
EFFORT_FEATURES = ["log_n_resources", "log_n_enemies", "log_n_mutualists",
                   "log_n_competitors", "log_total_interactions",
                   "log_globi_records", "log_n_pos_partners",
                   "log_n_neg_partners", "native_range_known"]
EFFORT_SCORE_FEATURES = ["log_globi_records", "log_total_interactions",
                         "log_n_pos_partners", "log_n_neg_partners"]
effort_idx = [sp_feature_names.index(n) for n in EFFORT_FEATURES]
effort_score = np.mean([sp_feat[n] for n in EFFORT_SCORE_FEATURES], axis=0)

# --- data support (exported, not a model input) ----------------------------
n_partner_units = np.maximum(n_total,
                             (n_pos_partners + n_neg_partners).numpy())
saturation = math.log1p(SUPPORT_SATURATION)
data_support = np.mean([
    np.minimum(1.0, np.log1p(n_range_locations) / saturation),
    np.minimum(1.0, np.log1p(n_partner_units) / saturation),
    native_known.astype(float)], axis=0)
SUPPORT_LABELS = ["low", "medium", "high"]
support_class = np.digitize(data_support, SUPPORT_CLASS_CUTS)

has_signal = (torch.as_tensor(n_total > 0)
              | presence.any(dim=1)
              | torch.as_tensor(native_known)
              | (n_pos_partners + n_neg_partners > 0))
eligible_mask = pool_mask & has_signal

# --- species information table ---------------------------------------------
FISH = {"Actinopterygii", "Chondrichthyes", "Elasmobranchii",
        "Cephalaspidomorphi", "Myxini", "Sarcopterygii", "Petromyzonti"}
HERP = {"Amphibia", "Reptilia", "Squamata", "Testudines", "Crocodylia"}


def frontend_group(kingdom_name, class_name):
    """Map taxonomy to the front-end group keys of the mockup."""
    if kingdom_name == "Plantae":
        return "plant"
    if kingdom_name != "Animalia":
        return "fungi"  # fungi and microbes (incl. Chromista, Bacteria)
    if class_name in FISH:
        return "fish"
    if class_name in HERP:
        return "herp"
    if class_name == "Aves":
        return "bird"
    if class_name == "Mammalia":
        return "mammal"
    return "invertebrate"


names = sp_tax["canonical_name"] if "canonical_name" in sp_tax else None
fallback_names = {}
if "gbif_canonical_name" in df_sinas.columns:
    fallback_names.update(df_sinas.drop_duplicates("gbif_id")
                          .set_index("gbif_id")["gbif_canonical_name"])
for side in ("source", "target"):
    col = f"{side}_taxon_name"
    if col in df_net.columns:
        for gid, nm in (df_net.drop_duplicates(f"{side}_gbif_id")
                        [[f"{side}_gbif_id", col]].itertuples(index=False)):
            fallback_names.setdefault(int(gid), nm)
taxon_names = [
    (names.iloc[i] if names is not None and pd.notna(names.iloc[i])
     else fallback_names.get(gid, f"GBIF:{gid}"))
    for i, gid in enumerate(all_species_ids)]

species_info = pd.DataFrame({
    "gbif_id": all_species_ids,
    "taxon_name": taxon_names,
    "kingdom": sp_tax["kingdom"].values,
    "phylum": sp_tax["phylum"].values,
    "class": sp_tax["class"].values,
    "order": sp_tax["order"].values,
    "family": sp_tax["family"].values,
    "n_alien_locations": n_alien_locations,
    "n_native_locations": n_native_locations,
    "n_range_locations": n_range_locations.astype(int),
    "n_partners": n_partner_units.astype(int),
    "n_pos_partners": n_pos_partners.numpy().astype(int),
    "n_neg_partners": n_neg_partners.numpy().astype(int),
    "native_range_known": native_known,
    "data_support": np.round(data_support, 3),
    "data_support_class": [SUPPORT_LABELS[i] for i in support_class],
    "effort_score": np.round(effort_score, 3),
    "gbif_fetch_failed": [gid in gbif_failed_ids for gid in all_species_ids],
})
species_info["group"] = [frontend_group(k, c) for k, c in
                         zip(kingdom, cls)]
eicat = {}
if EICAT_CSV.exists():
    eicat_df = pd.read_csv(EICAT_CSV)
    eicat = dict(zip(eicat_df["gbif_id"].astype(np.int64),
                     eicat_df["eicat"].astype(str).str.upper()))
species_info["eicat"] = species_info["gbif_id"].map(eicat).fillna("NE")

# names of taxonomy nodes (for the saliency maps)
taxon_node_names = {}
if "canonical_name" in tax.columns:
    taxon_node_names = dict(zip(tax["gbif_id"].dropna().astype(np.int64),
                                tax.loc[tax["gbif_id"].notna(),
                                        "canonical_name"]))
elig_np = eligible_mask.numpy()
print(f"   species features: {n_sp_features} ({len(EFFORT_FEATURES)} effort)"
      f" | with signal: {int(has_signal.sum()):,} | eligible (pool & "
      f"signal): {int(elig_np.sum()):,}")
print("   eligible species per data-support class: "
      + ", ".join(f"{lab} {int((support_class[elig_np] == j).sum()):,}"
                  for j, lab in enumerate(SUPPORT_LABELS)))
print(species_info.loc[elig_np, "group"].value_counts().to_string())

### 6c · Taxonomy node initialisation

Taxonomy nodes have no traits of their own. Each one starts as the mean
of its children's features (genus = mean of its species, family = mean of
its genera, …), projected to `EMB_DIM` with a fixed random matrix. This
gives the GNN an informative starting point instead of noise, and lets a
poorly documented species borrow information from its relatives.

`build_node_features` does this for any species feature matrix, because
the effort-neutral inputs of Step 9 need their own taxonomy nodes too.

In [ ]:
print("Step 6c: bottom-up taxonomy initialisation")
set_seed()
tax_link_index = []  # (child, parent, child rows, parent rows)
for child, parent in zip(LEVELS[:-1], LEVELS[1:]):
    links = tax_links[(child, parent)]
    c_idx = links["child"].map(node_index[child])
    p_idx = links["parent"].map(node_index[parent])
    ok = (c_idx.notna() & p_idx.notna()).to_numpy()
    tax_link_index.append((
        child, parent,
        torch.as_tensor(c_idx[ok].astype(np.int64).values),
        torch.as_tensor(p_idx[ok].astype(np.int64).values)))
taxonomy_projection = (torch.randn(n_sp_features, EMB_DIM)
                       / math.sqrt(n_sp_features))


def build_node_features(species_features):
    """GNN input features of every node type from species features.

    Taxonomy nodes get the mean of their children, walking up the ranks,
    projected with the fixed ``taxonomy_projection``. Works on the device
    of ``species_features``.
    """
    dev = species_features.device
    n_feat = species_features.shape[1]
    level = {"species": species_features}
    for child, parent, c_t, p_t in tax_link_index:
        acc = torch.zeros(num_nodes[parent], n_feat, device=dev)
        cnt = torch.zeros(num_nodes[parent], 1, device=dev)
        if len(c_t):
            c_t, p_t = c_t.to(dev), p_t.to(dev)
            acc.index_add_(0, p_t, level[child][c_t])
            cnt.index_add_(0, p_t, torch.ones(len(c_t), 1, device=dev))
        level[parent] = acc / cnt.clamp(min=1)
    projection = taxonomy_projection.to(dev)
    x = {"species": species_features}
    for rank in RANKS:
        if rank in node_types:
            x[rank] = level[rank] @ projection
    return x


x_dict = build_node_features(sp_features)
print("   " + ", ".join(f"{k}: {tuple(v.shape)}" for k, v in x_dict.items()))

## Step 7 · Model architecture

All classes live in one cell, so a fresh session can re-load a checkpoint
after running only Step 0 and this step.

| Class | Role |
|---|---|
| `CountryAutoencoder` | Compresses all location features into `CO_LATENT_DIM` numbers (pre-trained in Step 8, then frozen). |
| `RelationalEcologicalGNN` | 2-layer heterogeneous GraphSAGE over species and taxonomy, with DropEdge on interaction edges during training. Output: one `EMB_DIM` embedding per species. |
| `TradePathwayEncoder` | Fuses import strength of the target, trade routed into the target from the species' range, and the exporter strength of the range (with channel attention). |
| `InvasionRiskModel` | The seven streams and the classifier. Takes a dict of per-pair inputs ("leaves") and returns the logit and the climate gate. |
| `PairFeatureBuilder` | Turns *(species, location)* index pairs into those leaves, from static tensors plus one range matrix. Training, validation, scanning and the explanations all call it, so they all see identical inputs. |

**The leaves** (one row per pair):

| Leaf | Size | Stream |
|---|---|---|
| `sp_emb`, `co_latent` | `EMB_DIM`, `CO_LATENT_DIM` | A |
| `niche_mean`, `niche_std`, `target_climate` | `n_niche` each | B, C, gate |
| `koppen` | 3: profile overlap, coverage, data available | B |
| `trade_import`, `trade_export`, `trade_source` | channels, channels, aggs × channels | D |
| `biotic` | 6: helpful / harmful opportunity (shrunk), has helpful / harmful partners, log counts | E |
| `context` | `n_context` | F |
| `proximity` | 4: nearest-range closeness, neighbours, same-region share, range size | G |

**Pair-level leave-one-out.** All range-based leaves are computed from the
species' range *without the target location*. For unrecorded pairs
(negatives, scan candidates) this changes nothing; for positives it
removes the trivial "the target is part of the range" signal.

**Niche with shrinkage (new).** For a range of *n* locations with mean
*m* and sum of squared deviations *SS* on each climate axis:

$$\text{SD}^2 = \frac{SS + k \cdot \sigma^2_\text{typical}}{(n - 1) + k}$$

with *k* = `NICHE_PRIOR_STRENGTH` and σ²<sub>typical</sub> the median niche
variance of species with at least `NICHE_PRIOR_MIN_LOCATIONS` range
locations. One location gives the typical width, two give a width between
their spread and the typical one, many give their own spread. Before,
one location gave the *global* spread (every target looked suitable) and
two or three similar locations gave the 0.05 floor (every target looked
unsuitable). An empty range still gets the global mean and spread.

**Range thinning (new, training only).** `build(..., thin=True)` drops
a random share of each pair's range locations for `RANGE_THINNING_PROB`
of the pairs (at least one location stays), so the model also learns
from data-poor versions of well-known species. It is applied to
positives and negatives alike; otherwise "thin range" would mark one
class.

**Effort-neutral biotic part.** `build(..., neutral=True)` replaces the
species part of the biotic leaf (has / count of partners) by the values
of a typical known invader (Step 9).

**Climate gate.** A small network maps the Mahalanobis distance between
niche and target to a 0–1 suitability that scales the topology stream.
It is trained only by its own MSE loss against a Gaussian of that
distance (Step 10), and its output is detached inside the classifier, so
the invasion labels never shape it.

In [ ]:
LEAF_NAMES = ["sp_emb", "co_latent", "niche_mean", "niche_std",
              "target_climate", "koppen", "proximity", "context",
              "trade_import", "trade_export", "trade_source", "biotic"]
PAIR_LEAVES = [n for n in LEAF_NAMES if n != "sp_emb"]
TAXONOMY_PREFIXES = ("belongs_to_", "has_")  # edges kept by DropEdge


class CountryAutoencoder(nn.Module):
    """Compress the full location feature vector into a latent code."""

    def __init__(self, n_features, latent_dim, hidden_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, hidden_dim), nn.GELU(),
            nn.LayerNorm(hidden_dim), nn.Dropout(0.1),
            nn.Linear(hidden_dim, latent_dim), nn.GELU(),
            nn.LayerNorm(latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, n_features),
        )

    def forward(self, x):
        """Return (latent code, reconstruction)."""
        z = self.encoder(x)
        return z, self.decoder(z)

    def encode(self, x):
        """Latent code only."""
        return self.encoder(x)


class RelationalEcologicalGNN(nn.Module):
    """Two-layer heterogeneous GraphSAGE over species and taxonomy.

    Layer 1 aggregates direct neighbours, layer 2 two-hop structure. Each
    layer has its own convolutions. Node types without incoming messages
    keep their previous state. In training mode, ``edge_dropout`` removes
    that share of the interaction edges (DropEdge); taxonomy edges are
    always kept. The same reduced graph is used by both layers.
    """

    def __init__(self, dim, edge_types, node_types, n_sp_features,
                 dropout=0.4, feature_dropout=0.35, edge_dropout=0.0):
        super().__init__()
        self.sp_projector = nn.Linear(n_sp_features, dim)
        self.feature_dropout = nn.Dropout(feature_dropout)
        self.dropout = nn.Dropout(dropout)
        self.edge_dropout = float(edge_dropout)
        self.droppable = [et for et in edge_types
                          if not et[1].startswith(TAXONOMY_PREFIXES)]

        def schema():
            return {et: (SAGEConv(dim, dim) if et[0] == et[2]
                         else SAGEConv((dim, dim), dim))
                    for et in edge_types}

        self.conv1 = HeteroConv(schema(), aggr="mean")
        self.conv2 = HeteroConv(schema(), aggr="mean")
        self.norm1 = nn.ModuleDict({n: nn.LayerNorm(dim) for n in node_types})
        self.norm2 = nn.ModuleDict({n: nn.LayerNorm(dim) for n in node_types})

    def drop_edges(self, edge_index_dict):
        """Random subset of the interaction edges (training mode only)."""
        if not (self.training and self.edge_dropout > 0):
            return edge_index_dict
        out = dict(edge_index_dict)
        for et in self.droppable:
            if et in out:
                ei = out[et]
                keep = (torch.rand(ei.shape[1], device=ei.device)
                        >= self.edge_dropout)
                out[et] = ei[:, keep]
        return out

    def forward(self, x_dict, edge_index_dict):
        """Return a dict of node embeddings per node type."""
        edges = self.drop_edges(edge_index_dict)
        x = dict(x_dict)
        x["species"] = self.sp_projector(x["species"])
        if self.training:
            x = {k: self.feature_dropout(v) for k, v in x.items()}
        h = self.conv1(x, edges)
        h = {k: (self.norm1[k](self.dropout(F.gelu(h[k]))) if k in h
                 else x[k]) for k in x}
        h2 = self.conv2(h, edges)
        return {k: (self.norm2[k](F.gelu(h2[k])) if k in h2 else h[k])
                for k in h}


class TradePathwayEncoder(nn.Module):
    """Fuse import, routed export and source exporter strength.

    Channel attention (from import and routed export) weights the
    import x export interaction per channel. ``src_dim = 0`` drops the
    exporter-strength input.
    """

    def __init__(self, num_channels, latent_dim, hidden_dim=32, src_dim=0):
        super().__init__()
        self.src_dim = src_dim
        self.encoder = nn.Sequential(
            nn.Linear(num_channels * 3 + src_dim, hidden_dim), nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, latent_dim), nn.GELU(),
            nn.LayerNorm(latent_dim),
        )
        self.channel_attn = nn.Sequential(
            nn.Linear(num_channels * 2, num_channels), nn.Sigmoid())

    def forward(self, import_vec, export_vec, src_vec):
        """Return the trade latent vector per pair."""
        attn = self.channel_attn(torch.cat([import_vec, export_vec], -1))
        parts = [import_vec, export_vec, import_vec * export_vec * attn]
        if self.src_dim > 0:
            parts.append(src_vec)
        return self.encoder(torch.cat(parts, dim=-1))


class InvasionRiskModel(nn.Module):
    """Seven-stream classifier over the per-pair leaves.

    ``dims`` keys: sp_dim, co_latent_dim, n_niche, n_koppen, n_proximity,
    n_context, n_biotic, n_channels, src_dim, trade_latent_dim,
    topo_dropout. Stored in the checkpoint to rebuild the model.
    """

    def __init__(self, dims):
        super().__init__()
        self.dims = dict(dims)
        d = self.dims
        sp, nn_ = d["sp_dim"], d["n_niche"]
        self.topo_dropout = d.get("topo_dropout", 0.0)
        self.trade_encoder = TradePathwayEncoder(
            d["n_channels"], d["trade_latent_dim"], 32, d["src_dim"])
        self.topo_stream = nn.Sequential(
            nn.Linear(sp + d["co_latent_dim"], sp // 2), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(sp // 2, sp // 4), nn.GELU())
        self.niche_stream = nn.Sequential(
            nn.Linear(nn_ * 3 + d["n_koppen"], sp), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(sp, sp // 2), nn.GELU())
        self.suitability_stream = nn.Sequential(
            nn.Linear(nn_ * 2, nn_ * 4), nn.GELU(),
            nn.Linear(nn_ * 4, nn_), nn.Sigmoid())
        self.trade_proj = nn.Sequential(
            nn.Linear(d["trade_latent_dim"], sp // 4), nn.GELU())
        self.bio_proj = nn.Sequential(
            nn.Linear(d["n_biotic"], 16), nn.GELU(),
            nn.Linear(16, 8), nn.GELU())
        self.context_proj = nn.Sequential(
            nn.Linear(d["n_context"], 32), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(32, 8), nn.GELU())
        self.proximity_proj = nn.Sequential(
            nn.Linear(d["n_proximity"], 8), nn.GELU(),
            nn.Linear(8, 8), nn.GELU())
        self.climate_gate = nn.Sequential(
            nn.Linear(1 + nn_, nn_ * 2), nn.GELU(),
            nn.Linear(nn_ * 2, 1), nn.Sigmoid())
        combined = sp // 4 + sp // 2 + nn_ + sp // 4 + 8 + 8 + 8
        self.classifier = nn.Sequential(
            nn.Linear(combined, combined // 2), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(combined // 2, 1))

    @staticmethod
    def niche_distance(niche_mean, niche_std, target):
        """Per-axis z-difference and Mahalanobis (diagonal) distance."""
        z = (niche_mean - target) / niche_std
        return z, z.pow(2).sum(dim=-1, keepdim=True).sqrt()

    def gate(self, niche_mean, niche_std, target):
        """Climate suitability in [0, 1] from the niche distance."""
        z, m = self.niche_distance(niche_mean, niche_std, target)
        return self.climate_gate(torch.cat([m, z.pow(2)], dim=-1))

    def forward(self, x, detach_gate=True):
        """Return (logits, gate) for a dict of leaves (see LEAF_NAMES).

        ``detach_gate=False`` lets gradients flow through the gate; used
        only by the explanations.
        """
        z, _ = self.niche_distance(x["niche_mean"], x["niche_std"],
                                   x["target_climate"])
        gate = self.gate(x["niche_mean"], x["niche_std"],
                         x["target_climate"])
        topo = self.topo_stream(torch.cat([x["sp_emb"], x["co_latent"]],
                                          dim=-1))
        topo = topo * (gate.detach() if detach_gate else gate)
        if self.training and self.topo_dropout > 0:
            keep = (torch.rand(topo.shape[0], 1, device=topo.device)
                    > self.topo_dropout).float()
            topo = topo * keep
        niche = self.niche_stream(torch.cat(
            [x["niche_mean"], x["target_climate"], z.abs(), x["koppen"]],
            dim=-1))
        suit = self.suitability_stream(torch.cat(
            [x["niche_mean"], x["target_climate"]], dim=-1))
        trade = self.trade_proj(self.trade_encoder(
            x["trade_import"], x["trade_export"], x["trade_source"]))
        bio = self.bio_proj(x["biotic"])
        ctx = self.context_proj(x["context"])
        prox = self.proximity_proj(x["proximity"])
        logits = self.classifier(torch.cat(
            [topo, niche, suit, trade, bio, ctx, prox], dim=-1)).squeeze(-1)
        return logits, gate.squeeze(-1)


class PairFeatureBuilder:
    """Build the per-pair leaves from static tensors and a range matrix.

    ``static`` keys: co_niche, co_koppen, co_koppen_available, co_context,
    co_latent, dist_km, same_region, trade, co_import, co_export, pos_opp,
    neg_opp, pos_prior, neg_prior, n_pos, n_neg, sp_bio_static,
    n_locations, proximity_scale_km, neighbour_radius_km, niche_std_floor,
    niche_prior_strength, niche_prior_min_locations,
    biotic_prior_strength, thin_prob, thin_keep_min.

    ``range_mat`` (S x N) marks each species' known range for this split.
    ``bio_neutral`` (S x 4) is the effort-neutral species part of the
    biotic leaf, used by ``build(..., neutral=True)``.
    """

    def __init__(self, static, range_mat, src_aggs=("mean", "max"),
                 chunk=4096, bio_neutral=None):
        self.s = static
        self.src_aggs = tuple(src_aggs)
        self.chunk = chunk
        self.P = range_mat.float()
        self.bio_neutral = (static["sp_bio_static"] if bio_neutral is None
                            else bio_neutral)
        clim = static["co_niche"]
        self.g_mean = clim.mean(dim=0, keepdim=True)
        self.g_var = clim.var(dim=0, unbiased=False,
                              keepdim=True).clamp(min=1e-8)
        self.prior_var = self._typical_niche_variance(clim)

    def _typical_niche_variance(self, clim):
        """Median per-axis niche variance of species with a large range."""
        n = self.P.sum(dim=1)
        big = n >= self.s["niche_prior_min_locations"]
        if int(big.sum()) < 10:  # tiny data sets: relax the threshold
            big = n >= 3
        if int(big.sum()) == 0:
            return self.g_var.clone()
        rows, nb = self.P[big], n[big].unsqueeze(1)
        mean = (rows @ clim) / nb
        var = ((rows @ clim.pow(2)) / nb - mean.pow(2)).clamp(min=0)
        return var.median(dim=0, keepdim=True).values.clamp(min=1e-6)

    def range_mask(self, sp, co, loo=True):
        """(b, N) copy of the range rows, target left out if ``loo``."""
        mask = self.P[sp].clone()
        if loo:
            mask[torch.arange(len(co), device=mask.device), co] = 0.0
        return mask

    def thin(self, mask):
        """Drop a random share of range locations (training augmentation).

        A share ``thin_prob`` of the rows keeps each location with a
        probability drawn from [thin_keep_min, 1]; a non-empty range keeps
        at least one location.
        """
        prob, keep_min = self.s["thin_prob"], self.s["thin_keep_min"]
        if prob <= 0:
            return mask
        b, dev = mask.shape[0], mask.device
        applied = torch.rand(b, 1, device=dev) < prob
        frac = keep_min + (1.0 - keep_min) * torch.rand(b, 1, device=dev)
        r = torch.rand_like(mask)
        out = mask * ((r < frac) | ~applied).float()
        lost = (out.sum(dim=1) == 0) & (mask.sum(dim=1) > 0)
        if bool(lost.any()):
            rows = lost.nonzero(as_tuple=True)[0]
            cols = ((r[rows] + 1e-6) * mask[rows]).argmax(dim=1)
            out[rows, cols] = 1.0
        return out

    def niche(self, sp, co, loo=True, mask=None):
        """Realised climate niche (mean, SD) over the range, shrunk.

        SD^2 = (SS + k * typical variance) / (n - 1 + k); an empty range
        gets the global mean and spread (see the markdown above).
        """
        if mask is None:
            mask = self.range_mask(sp, co, loo)
        clim = self.s["co_niche"]
        n = mask.sum(dim=1, keepdim=True)
        s, q = mask @ clim, mask @ clim.pow(2)
        has = n > 0.5
        mean = torch.where(has, s / n.clamp(min=1), self.g_mean)
        ss = (q - n * mean.pow(2)).clamp(min=0)
        k = self.s["niche_prior_strength"]
        var = torch.where(has, (ss + k * self.prior_var)
                          / ((n - 1).clamp(min=0) + k), self.g_var)
        return mean, var.sqrt().clamp(min=self.s["niche_std_floor"])

    def koppen(self, sp, co, loo=True, mask=None):
        """[profile overlap, class coverage, data available], (b, 3).

        overlap  = sum over classes of min(range profile, target share)
        coverage = target area share in classes seen anywhere in the range
        Range locations without Koppen-Geiger data are left out of the
        profile; without data on either side all three are 0.
        """
        if mask is None:
            mask = self.range_mask(sp, co, loo)
        kg = self.s["co_koppen"]
        avail = self.s["co_koppen_available"].float()
        tgt = kg[co]
        n_kg = mask @ avail
        profile = (mask @ kg) / n_kg.clamp(min=1).unsqueeze(1)
        seen = ((mask @ (kg > 0).float()) > 0.5).float()
        has = ((n_kg > 0.5) & (avail[co] > 0.5)).float()
        overlap = torch.minimum(profile, tgt).sum(dim=1) * has
        coverage = (tgt * seen).sum(dim=1) * has
        return torch.stack([overlap, coverage, has], dim=1)

    def proximity(self, sp, co, loo=True, mask=None):
        """[closeness of nearest range location, log #range locations
        within the radius (scaled), share of range in the same region,
        log range size (scaled)], (b, 4)."""
        if mask is None:
            mask = self.range_mask(sp, co, loo)
        d = self.s["dist_km"][co]
        nearest = torch.where(mask > 0, d, torch.full_like(d, float("inf")))
        closeness = torch.exp(-nearest.amin(dim=1)
                              / self.s["proximity_scale_km"])
        near = (mask * (d <= self.s["neighbour_radius_km"]).float()).sum(1)
        size = mask.sum(dim=1)
        same = ((mask * self.s["same_region"][co].float()).sum(1)
                / size.clamp(min=1))
        scale = math.log1p(self.s["n_locations"])
        return torch.stack([closeness, torch.log1p(near) / scale, same,
                            torch.log1p(size) / scale], dim=1)

    def trade(self, sp, co, loo=True, mask=None):
        """Import strength of the target, trade routed into the target from
        the range, and exporter strength of the range per aggregation."""
        if mask is None:
            mask = self.range_mask(sp, co, loo)
        n_src = mask.sum(dim=1, keepdim=True).clamp(min=1)
        to_dest = self.s["trade"][:, :, co].permute(2, 0, 1)  # (b, C, N)
        export = (to_dest * mask.unsqueeze(1)).sum(dim=2) / n_src
        exporter = self.s["co_export"]
        cols = []
        for agg in self.src_aggs:
            if agg == "mean":
                cols.append((mask @ exporter) / n_src)
            elif agg == "max":
                cols.append((mask.unsqueeze(2) * exporter.unsqueeze(0))
                            .amax(dim=1))
            else:
                raise ValueError(f"Unknown exporter-strength agg: {agg}")
        src = (torch.cat(cols, dim=1) if cols
               else mask.new_zeros(len(sp), 0))
        return self.s["co_import"][co], export, src

    def biotic(self, sp, co, neutral=False):
        """[pos. opportunity, neg. opportunity, has pos, has neg,
        log #pos, log #neg], (b, 6). Opportunities are shrunk (Step 5)."""
        opp = torch.stack([self.s["pos_opp"][sp, co],
                           self.s["neg_opp"][sp, co]], dim=1)
        species_part = (self.bio_neutral if neutral
                        else self.s["sp_bio_static"])[sp]
        return torch.cat([opp, species_part], dim=1)

    def raw_opportunity(self, sign, sp, co):
        """Observed (unshrunk) partner share, for the exports."""
        shrunk = self.s[f"{sign}_opp"][sp, co]
        n = self.s[f"n_{sign}"][sp]
        k = self.s["biotic_prior_strength"]
        prior = self.s[f"{sign}_prior"][co]
        raw = ((n + k) * shrunk - k * prior) / n.clamp(min=1)
        return torch.where(n > 0, raw, torch.zeros_like(raw)).clamp(0, 1)

    @torch.no_grad()
    def build(self, sp, co, loo=True, neutral=False, thin=False):
        """All leaves except ``sp_emb`` for index tensors ``sp``, ``co``.

        ``neutral``: effort-neutral species part of the biotic leaf.
        ``thin``: range-thinning augmentation (training pairs only).
        """
        if len(sp) == 0:
            raise ValueError("build() called with no pairs")
        parts = defaultdict(list)
        for i in range(0, len(sp), self.chunk):
            s, c = sp[i:i + self.chunk], co[i:i + self.chunk]
            mask = self.range_mask(s, c, loo)
            if thin:
                mask = self.thin(mask)
            mean, std = self.niche(s, c, mask=mask)
            imp, exp, src = self.trade(s, c, mask=mask)
            values = {
                "co_latent": self.s["co_latent"][c],
                "niche_mean": mean, "niche_std": std,
                "target_climate": self.s["co_niche"][c],
                "koppen": self.koppen(s, c, mask=mask),
                "proximity": self.proximity(s, c, mask=mask),
                "context": self.s["co_context"][c],
                "trade_import": imp, "trade_export": exp,
                "trade_source": src,
                "biotic": self.biotic(s, c, neutral),
            }
            for key, value in values.items():
                parts[key].append(value)
        return {k: torch.cat(v) for k, v in parts.items()}

    @torch.no_grad()
    def describe(self, sp, co):
        """Readable raw values per pair (for the export, not the model)."""
        mask = self.range_mask(sp, co, loo=True)
        mean, std = self.niche(sp, co, mask=mask)
        _, mdist = InvasionRiskModel.niche_distance(
            mean, std, self.s["co_niche"][co])
        d = self.s["dist_km"][co]
        nearest = torch.where(mask > 0, d, torch.full_like(d, float("inf")))
        return {
            "feat_mahalanobis_distance": mdist.squeeze(1),
            "feat_koppen_overlap": self.koppen(sp, co, mask=mask)[:, 0],
            "feat_nearest_range_km": nearest.amin(dim=1),
            "feat_n_range_locations": mask.sum(dim=1),
            "feat_biotic_opp_positive": self.raw_opportunity("pos", sp, co),
            "feat_biotic_opp_negative": self.raw_opportunity("neg", sp, co),
        }


def with_embedding(x, sp, sp_emb):
    """Add the species embedding rows of ``sp`` to a dict of leaves."""
    return {**x, "sp_emb": sp_emb[sp]}


def take(x, idx):
    """Index every leaf of a dict with the same row index."""
    return {k: v[idx] for k, v in x.items()}


@torch.no_grad()
def score_pairs(model, builder, sp, co, sp_emb, neutral=True, chunk=16384):
    """Logits and gate values for many pairs, chunked, eval mode.

    ``neutral=True`` is the scan mode (effort-neutral biotic part); use
    the matching ``sp_emb`` (from the effort-neutral node features).
    """
    model.eval()
    logits, gates = [], []
    for i in range(0, len(sp), chunk):
        s, c = sp[i:i + chunk], co[i:i + chunk]
        lg, g = model(with_embedding(builder.build(s, c, neutral=neutral),
                                     s, sp_emb))
        logits.append(lg)
        gates.append(g)
    return torch.cat(logits), torch.cat(gates)


@torch.no_grad()
def ensemble_logits(members, builder, sp, co, raw=False, chunk=16384):
    """Mean logit over the ensemble members, chunked, eval mode.

    Each member is a dict with ``model``, ``sp_emb`` (scan mode) and
    ``sp_emb_raw`` (actual effort features). ``raw=True`` scores with the
    actual effort features (only meaningful with "neutralize").
    """
    key = "sp_emb_raw" if raw else "sp_emb"
    out = []
    for i in range(0, len(sp), chunk):
        s, c = sp[i:i + chunk], co[i:i + chunk]
        leaves = builder.build(s, c, neutral=not raw)
        total = 0.0
        for m in members:
            m["model"].eval()
            total = total + m["model"](with_embedding(leaves, s,
                                                      m[key]))[0]
        out.append(total / len(members))
    return torch.cat(out)

## Step 8 · Pre-train the location autoencoder

The autoencoder learns to reconstruct all ≈100 z-scored location features
from `CO_LATENT_DIM` numbers. 15 % of the locations are held out to stop
training when the reconstruction error on unseen locations stops
improving; the best weights are then used to encode **all** locations.
The latent code is frozen afterwards and feeds the topology stream.

In [ ]:
print("Step 8: location autoencoder")
set_seed()
perm = torch.randperm(num_countries)
n_val_ae = max(1, int(AE_VAL_FRACTION * num_countries))
ae_val, ae_train = co_all[perm[:n_val_ae]], co_all[perm[n_val_ae:]]
country_ae = CountryAutoencoder(n_co_features, CO_LATENT_DIM)
ae_opt = torch.optim.Adam(country_ae.parameters(), lr=1e-3,
                          weight_decay=1e-5)
best_ae = {"loss": float("inf"), "epoch": 0,
           "state": copy.deepcopy(country_ae.state_dict())}
patience = 0
for epoch in range(1, AE_MAX_EPOCHS + 1):
    country_ae.train()
    ae_opt.zero_grad()
    loss = F.mse_loss(country_ae(ae_train)[1], ae_train)
    loss.backward()
    ae_opt.step()
    country_ae.eval()
    with torch.no_grad():
        val_loss = F.mse_loss(country_ae(ae_val)[1], ae_val).item()
    if val_loss < best_ae["loss"] - 1e-6:
        best_ae = {"loss": val_loss, "epoch": epoch,
                   "state": copy.deepcopy(country_ae.state_dict())}
        patience = 0
    else:
        patience += 1
        if patience >= AE_PATIENCE:
            break
    if epoch % 100 == 0:
        print(f"   epoch {epoch:4d} | train {loss.item():.4f} | "
              f"val {val_loss:.4f}")
country_ae.load_state_dict(best_ae["state"])
country_ae.eval()
with torch.no_grad():
    co_latent = country_ae.encode(co_all)
print(f"   best epoch {best_ae['epoch']} | val reconstruction MSE "
      f"{best_ae['loss']:.4f} (z units; 1.0 = no better than the mean) | "
      f"latent {tuple(co_latent.shape)}")

## Step 9 · Device setup, matched sampling, effort views, ranking metric

Everything moves to `DEVICE` once.

**Positive weights.** Each positive pair gets weight
*n*<sup>−`POSITIVE_WEIGHT_POWER`</sup>, with *n* the species' positives in
the training split, scaled to mean 1. A species alien in 100 locations
then weighs 10× (not 100×) a species alien in one location.

**Matched negatives (`NegativeSampler`).** Negatives are unlabeled pairs
outside `exclude_mask`. Their species are drawn so that the strata in
`MATCH_NEGATIVES_ON` — front-end group and/or effort bin
(`EFFORT_BINS`: bin 0 = no GloBI documentation, then quantile bins) —
have the same (weighted) distribution as the positives' species; within
a stratum every pool species is equally likely. `NEG_LOCATION_MATCHED`
of them are placed in the locations of the positives, the rest uniformly.
Then neither documentation level, group nor location recording on its
own separates positives from negatives, and the model has to use the
pair-level evidence.

*Hard* negatives are drawn from the same sampler and kept when they are
climatically close (20th–60th percentile of the niche distance). The
closest tier is skipped on purpose: in a PU setting it overlaps most with
undetected invasions.

**Effort views (`effort_views`).** For each split three inputs are made:

| View | Used for | Effort features |
|---|---|---|
| `x_train` | GNN input during training | actual values (`"neutralize"`, `"none"`) or typical values (`"drop"`) |
| `x_scan` | GNN input for validation, scanning and explanations | typical values (`"neutralize"`, `"drop"`) or actual (`"none"`) |
| `bio_neutral` | species part of the biotic leaf in scan mode | typical values |

"Typical" = the median over the species of the training positives (for
production: over all positive species). Scores are therefore read as
"risk if this species were documented like a typical known invader".

**Ranking metric.** AUC compares positives with random unlabeled pairs;
a horizon scan instead ranks all candidates *of one location*. For each
location with held-out positives, every held-out positive is ranked
among a sample of that location's candidates. Reported: the median
position (as a fraction from the top) and the share in the top 5 % and
10 %, overall and **per data-support class** of the held-out species
(`…_support_low/medium/high`). If the correction works, the low-support
classes should not lag far behind. It is a lower bound, because the
candidates include undetected invasions.

In [ ]:
print("Step 9: device setup and sampling")
set_seed()
static = {
    "co_niche": co_niche, "co_koppen": co_koppen,
    "co_koppen_available": co_koppen_available, "co_context": co_context,
    "co_latent": co_latent, "dist_km": dist_km, "same_region": same_region,
    "trade": trade_tensor, "co_import": co_trade_import,
    "co_export": co_trade_export, "pos_opp": pos_opp_mat,
    "neg_opp": neg_opp_mat, "pos_prior": pos_prior, "neg_prior": neg_prior,
    "n_pos": n_pos_partners, "n_neg": n_neg_partners,
    "sp_bio_static": sp_bio_static,
    "n_locations": num_countries,
    "proximity_scale_km": PROXIMITY_SCALE_KM,
    "neighbour_radius_km": NEIGHBOUR_RADIUS_KM,
    "niche_std_floor": NICHE_STD_FLOOR,
    "niche_prior_strength": NICHE_PRIOR_STRENGTH,
    "niche_prior_min_locations": NICHE_PRIOR_MIN_LOCATIONS,
    "biotic_prior_strength": BIOTIC_PRIOR_STRENGTH,
    "thin_prob": RANGE_THINNING_PROB,
    "thin_keep_min": RANGE_THINNING_KEEP_MIN,
}
static = {k: (v.to(DEVICE) if torch.is_tensor(v) else v)
          for k, v in static.items()}
base_range_mat = base_range_mat.to(DEVICE)
exclude_mask = exclude_mask.to(DEVICE)
eligible_mask = eligible_mask.to(DEVICE)
eligible_species = eligible_mask.nonzero(as_tuple=True)[0]
pos_sp_all, pos_co_all = pos_sp_all.to(DEVICE), pos_co_all.to(DEVICE)
sp_features = sp_features.to(DEVICE)
x_dict_raw = build_node_features(sp_features)
edge_index_dev = {et: ei.to(DEVICE) for et, ei in edge_index_dict.items()}
effort_idx_t = torch.as_tensor(effort_idx, device=DEVICE)
support_class_t = torch.as_tensor(support_class, device=DEVICE)

model_dims = dict(
    sp_dim=EMB_DIM, co_latent_dim=CO_LATENT_DIM, n_niche=n_niche,
    n_koppen=3, n_proximity=4, n_context=n_context, n_biotic=6,
    n_channels=num_channels, src_dim=SRC_DIM,
    trade_latent_dim=TRADE_LATENT_DIM, topo_dropout=TOPO_STREAM_DROPOUT)
print(f"   device: {DEVICE} | eligible species: {len(eligible_species):,}")

# --- strata for matched negative sampling --------------------------------
effort_bin = np.zeros(num_species, dtype=np.int64)
documented = effort_score > 0
ref_scores = effort_score[documented & eligible_mask.cpu().numpy()]
if len(ref_scores):
    cuts = np.quantile(ref_scores, np.linspace(0, 1, EFFORT_BINS)[1:-1])
    effort_bin[documented] = 1 + np.digitize(effort_score[documented], cuts)
group_code = pd.factorize(species_info["group"])[0].astype(np.int64)
strata = np.zeros(num_species, dtype=np.int64)
if "group" in MATCH_NEGATIVES_ON:
    strata = strata * (group_code.max() + 1) + group_code
if "effort" in MATCH_NEGATIVES_ON:
    strata = strata * (effort_bin.max() + 1) + effort_bin
species_strata = (torch.as_tensor(strata, device=DEVICE)
                  if MATCH_NEGATIVES_ON else None)
print(f"   negative strata: "
      f"{len(np.unique(strata[eligible_mask.cpu().numpy()]))} "
      f"({' x '.join(MATCH_NEGATIVES_ON) or 'none: uniform species'})")


def positive_weights(sp):
    """Per-pair weights (mean 1) that down-weight species with many
    positives in this split (``POSITIVE_WEIGHT_POWER``)."""
    if POSITIVE_WEIGHT_POWER == 0 or len(sp) == 0:
        return torch.ones(len(sp), device=sp.device)
    counts = torch.bincount(sp, minlength=num_species).float()[sp]
    w = counts.pow(-POSITIVE_WEIGHT_POWER)
    return w / w.mean()


def sample_unlabeled(n, sp_pool):
    """``n`` uniform random pairs from ``sp_pool`` x all locations, never
    recorded. Used for the uniform AUC, the IG baseline and checks."""
    empty = torch.empty(0, dtype=torch.long, device=DEVICE)
    if n <= 0:
        return empty, empty
    out_sp, out_co, got = [], [], 0
    for _ in range(100):
        m = max(3 * n, 1024)
        sp = sp_pool[torch.randint(len(sp_pool), (m,), device=DEVICE)]
        co = torch.randint(num_countries, (m,), device=DEVICE)
        ok = ~exclude_mask[sp, co]
        out_sp.append(sp[ok])
        out_co.append(co[ok])
        got += int(ok.sum())
        if got >= n:
            return torch.cat(out_sp)[:n], torch.cat(out_co)[:n]
    raise RuntimeError("Could not draw enough unlabeled pairs.")


class NegativeSampler:
    """Unlabeled pairs that resemble the positives in all but the label.

    Species: the strata of ``species_strata`` get the same weighted share
    as among the positives; every pool species of a stratum is equally
    likely (uniform over the pool if there are no strata, or as a
    fallback). Locations: a share ``location_matched`` follows the
    positives' (weighted) locations, the rest is uniform. Recorded pairs
    (``exclude_mask``) are never returned.
    """

    def __init__(self, pos_sp, pos_co, pos_w, pool,
                 location_matched=NEG_LOCATION_MATCHED):
        in_pool = torch.zeros(num_species, dtype=torch.bool, device=DEVICE)
        in_pool[pool] = True
        prob = in_pool.float()
        if species_strata is not None and len(pos_sp):
            n_strata = int(species_strata.max()) + 1
            target = torch.zeros(n_strata, device=DEVICE).index_add_(
                0, species_strata[pos_sp], pos_w.float())
            count = torch.bincount(species_strata[in_pool],
                                   minlength=n_strata).float()
            per_species = target / count.clamp(min=1)
            matched = per_species[species_strata] * in_pool.float()
            if float(matched.sum()) > 0:
                prob = matched
        self.sp_prob = prob / prob.sum()
        loc = torch.zeros(num_countries, device=DEVICE)
        if len(pos_co):
            loc = loc.index_add_(0, pos_co, pos_w.float())
        self.loc_prob = (loc / loc.sum() if float(loc.sum()) > 0
                         else torch.full((num_countries,),
                                         1.0 / num_countries,
                                         device=DEVICE))
        self.location_matched = float(location_matched)

    def sample(self, n):
        """``n`` (species, location) index pairs."""
        empty = torch.empty(0, dtype=torch.long, device=DEVICE)
        if n <= 0:
            return empty, empty
        out_sp, out_co, got = [], [], 0
        for _ in range(100):
            m = max(3 * n, 1024)
            sp = inverse_cdf_sample(self.sp_prob, m)
            co = torch.randint(num_countries, (m,), device=DEVICE)
            if self.location_matched > 0:
                use = torch.rand(m, device=DEVICE) < self.location_matched
                co = torch.where(use, inverse_cdf_sample(self.loc_prob, m),
                                 co)
            ok = ~exclude_mask[sp, co]
            out_sp.append(sp[ok])
            out_co.append(co[ok])
            got += int(ok.sum())
            if got >= n:
                return torch.cat(out_sp)[:n], torch.cat(out_co)[:n]
        raise RuntimeError("Could not draw enough matched unlabeled pairs.")


def sample_hard(n, builder, sampler, lo_pct=20, hi_pct=60):
    """Climatically close negatives drawn from a matched sampler."""
    cand_sp, cand_co = sampler.sample(n * 8)
    if n <= 0:
        return cand_sp, cand_co
    mean, std = builder.niche(cand_sp, cand_co)
    _, dist = InvasionRiskModel.niche_distance(mean, std,
                                               static["co_niche"][cand_co])
    order = torch.argsort(dist.squeeze(1))
    band = order[int(len(order) * lo_pct / 100):
                 int(len(order) * hi_pct / 100)]
    if len(band) < n:
        band = order[:max(n, len(band))]
    pick = band[torch.randperm(len(band), device=DEVICE)[:n]]
    return cand_sp[pick], cand_co[pick]


def effort_views(ref_species):
    """(x_train, x_scan, bio_neutral) for one split; see the table above.

    ``ref_species``: species whose median effort values count as typical.
    """
    if EFFORT_CORRECTION == "none":
        return x_dict_raw, x_dict_raw, static["sp_bio_static"]
    ref = torch.unique(ref_species)
    feats = sp_features.clone()
    feats[:, effort_idx_t] = (sp_features[ref][:, effort_idx_t]
                              .median(dim=0).values)
    x_scan = build_node_features(feats)
    bio = static["sp_bio_static"]
    bio_neutral = (bio[ref].median(dim=0).values.unsqueeze(0)
                   .expand_as(bio).contiguous())
    x_train = x_scan if EFFORT_CORRECTION == "drop" else x_dict_raw
    return x_train, x_scan, bio_neutral


def weighted_metrics(pos_scores, neg_scores, pos_w):
    """ROC-AUC, PR-AUC and curves with per-species positive weights."""
    y = np.r_[np.ones(len(pos_scores)), np.zeros(len(neg_scores))]
    s = np.r_[pos_scores, neg_scores]
    w = np.r_[pos_w, np.ones(len(neg_scores))]
    fpr, tpr, _ = roc_curve(y, s, sample_weight=w)
    precision, recall, _ = precision_recall_curve(y, s, sample_weight=w)
    return (roc_auc_score(y, s, sample_weight=w), auc(recall, precision),
            fpr, tpr, precision, recall)


@torch.no_grad()
def ranking_eval(score_fn, val_sp, val_co, cand_mask,
                 n_candidates=RANK_EVAL_CANDIDATES,
                 max_locations=RANK_EVAL_MAX_LOCATIONS, seed=0):
    """Where do held-out positives land in their location's scan list?

    ``score_fn(sp, co)`` returns logits. Results overall and per
    data-support class of the held-out species.
    """
    gen = torch.Generator().manual_seed(seed)
    locations = val_co.unique().cpu()
    if len(locations) > max_locations:
        locations = locations[torch.randperm(len(locations),
                                             generator=gen)[:max_locations]]
    top_fracs, classes = [], []
    for c in locations.tolist():
        pos = val_sp[val_co == c]
        cand = (cand_mask & ~exclude_mask[:, c]).nonzero(as_tuple=True)[0]
        if len(pos) == 0 or len(cand) == 0:
            continue
        if len(cand) > n_candidates:
            keep = torch.randperm(len(cand), generator=gen)[:n_candidates]
            cand = cand[keep.to(cand.device)]
        s_pos = score_fn(pos, torch.full_like(pos, c))
        s_cand = score_fn(cand, torch.full_like(cand, c))
        below = torch.searchsorted(s_cand.sort().values.contiguous(),
                                   s_pos.contiguous()).float()
        top_fracs.append((1.0 - below / len(cand)).cpu())
        classes.append(support_class_t[pos].cpu())
    if not top_fracs:
        return {}
    tf = torch.cat(top_fracs).numpy()
    cl = torch.cat(classes).numpy()
    out = {"rank_median_top_frac": float(np.median(tf)),
           "rank_hit_top5pct": float((tf <= 0.05).mean()),
           "rank_hit_top10pct": float((tf <= 0.10).mean()),
           "rank_n_pos": int(len(tf)),
           "rank_n_locations": len(top_fracs)}
    for j, lab in enumerate(SUPPORT_LABELS):
        sel = cl == j
        out[f"rank_n_pos_support_{lab}"] = int(sel.sum())
        if sel.any():
            out[f"rank_median_top_frac_support_{lab}"] = float(
                np.median(tf[sel]))
            out[f"rank_hit_top10pct_support_{lab}"] = float(
                (tf[sel] <= 0.10).mean())
    return out

## Step 10 · Train and evaluate one split

`train_and_evaluate_split` is used for every cross-validation fold and
for each production model.

1. **Range for this split:** `base_range_mat` with the held-out positives
   removed, so no validation pair feeds any feature.
2. **Effort views** (Step 9) from the species of the training positives
   (production: all positive species), and **positive weights**.
3. **Fixed validation set:** held-out positives and *matched* unlabeled
   pairs from the validation species pool, both in scan mode
   (effort-neutral). Checkpoints are therefore chosen on what the
   deployed scanner sees, and a model cannot be picked for exploiting
   documentation.
4. **Each epoch**
   * (re)sample negatives every `NEG_REFRESH_EVERY` epochs: half hard,
     half easy, both matched; re-thin the ranges of positives and
     negatives (`RANGE_THINNING_PROB`);
   * *BCE step:* GNN (with DropEdge) + classifier on positives vs
     negatives, label smoothing, **weighted** positives;
   * *gate step:* the climate gate alone, MSE against exp(−d² / 2σ²) of
     the Mahalanobis distance d (σ = median d of the training
     positives), on positives and easy negatives;
   * *validation:* weighted ROC-AUC and the gate separation (mean gate of
     positives − unlabeled).
5. **Checkpoint** on AUC + `GATE_SCORE_WEIGHT` × gate separation (kept in
   memory); early stopping after `EARLY_STOP_PATIENCE` epochs without
   improvement.
6. **Final metrics** on the restored best model:

| Metric | Negatives | Scoring | Meaning |
|---|---|---|---|
| `roc_auc`, `pr_auc` | matched | scan mode | main metric; per-species weighted |
| `roc_auc_uniform_negatives` | uniform (as in version 2) | scan mode | comparable with version 2 |
| `roc_auc_effort_raw` | matched | actual effort features | how much documentation alone would add |
| ranking metric | all candidates of a location | scan mode | the horizon-scan task, also per data-support class |

In [ ]:
def train_and_evaluate_split(train_idx, val_idx, label, train_pool=None,
                             val_pool=None, rank_eval=True, verbose=True,
                             seed=SEED, effort_ref=None):
    """Train on positives ``train_idx``, validate on ``val_idx``.

    ``train_idx`` / ``val_idx`` index ``pos_sp_all`` / ``pos_co_all``.
    ``train_pool`` / ``val_pool`` restrict the species negatives are
    drawn from (default: all eligible species). ``effort_ref`` (species
    indices) defines the typical effort values for scoring (default: the
    species of the training positives). Returns a result dict with
    metrics, the trained modules, the split's feature builder and its
    effort views.
    """
    set_seed(seed)
    train_pool = eligible_species if train_pool is None else train_pool
    val_pool = eligible_species if val_pool is None else val_pool
    train_sp, train_co = pos_sp_all[train_idx], pos_co_all[train_idx]
    val_sp, val_co = pos_sp_all[val_idx], pos_co_all[val_idx]
    w_train, w_val = positive_weights(train_sp), positive_weights(val_sp)
    x_train_dict, x_scan_dict, bio_neutral = effort_views(
        train_sp if effort_ref is None else effort_ref)
    train_neutral = EFFORT_CORRECTION == "drop"
    thin = RANGE_THINNING_PROB > 0

    encoder = RelationalEcologicalGNN(EMB_DIM, edge_types, node_types,
                                      n_sp_features,
                                      edge_dropout=EDGE_DROPOUT).to(DEVICE)
    model = InvasionRiskModel(model_dims).to(DEVICE)
    range_mat = base_range_mat.clone()
    range_mat[val_sp, val_co] = 0.0
    builder = PairFeatureBuilder(static, range_mat, SRC_STRENGTH_AGGS,
                                 FEATURE_CHUNK, bio_neutral=bio_neutral)
    train_sampler = NegativeSampler(train_sp, train_co, w_train, train_pool)
    val_sampler = NegativeSampler(val_sp, val_co, w_val, val_pool)

    with torch.no_grad():
        sample = torch.randperm(len(train_sp), device=DEVICE)[:2000]
        mean, std = builder.niche(train_sp[sample], train_co[sample])
        _, dist = InvasionRiskModel.niche_distance(
            mean, std, static["co_niche"][train_co[sample]])
        gate_sigma = max(float(dist.median()), 1e-3)

    val_neg_sp, val_neg_co = val_sampler.sample(len(val_sp) * VAL_NEG_RATIO)
    x_val_pos = builder.build(val_sp, val_co, neutral=True)
    x_val_neg = builder.build(val_neg_sp, val_neg_co, neutral=True)
    w_val_np = w_val.cpu().numpy()
    y_val = np.r_[np.ones(len(val_sp)), np.zeros(len(val_neg_sp))]
    sw_val = np.r_[w_val_np, np.ones(len(val_neg_sp))]
    x_train_pos = builder.build(train_sp, train_co, neutral=train_neutral)

    gate_params = list(model.climate_gate.parameters())
    gate_ids = {id(p) for p in gate_params}
    main_params = (list(encoder.parameters())
                   + [p for p in model.parameters() if id(p) not in gate_ids])
    optimizer = torch.optim.Adam(main_params, lr=LEARNING_RATE,
                                 weight_decay=WEIGHT_DECAY)
    gate_opt = torch.optim.Adam(gate_params, lr=5e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6)
    gate_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        gate_opt, mode="min", factor=0.5, patience=10, min_lr=1e-6)

    best = {"score": -float("inf"), "epoch": 0, "auc": float("nan"),
            "gate_sep": float("nan"),
            "encoder": copy.deepcopy(encoder.state_dict()),
            "model": copy.deepcopy(model.state_dict())}
    patience, history = 0, []
    n_hard = len(train_sp) * (TRAIN_NEG_RATIO // 2)
    n_easy = len(train_sp) * TRAIN_NEG_RATIO - n_hard
    if verbose:
        print(f"\n[{label}] train {len(train_sp):,} | val {len(val_sp):,} "
              f"| gate sigma {gate_sigma:.3f}")

    for epoch in range(1, N_EPOCHS + 1):
        if (epoch - 1) % NEG_REFRESH_EVERY == 0:
            h_sp, h_co = sample_hard(n_hard, builder, train_sampler)
            e_sp, e_co = train_sampler.sample(n_easy)
            neg_sp, neg_co = torch.cat([h_sp, e_sp]), torch.cat([h_co, e_co])
            n_easy_now = len(e_sp)
            check(not exclude_mask[neg_sp, neg_co].any(),
                  "a training negative is a recorded pair")
            x_neg = builder.build(neg_sp, neg_co, neutral=train_neutral,
                                  thin=thin)
            if thin:  # positives get freshly thinned ranges as well
                x_train_pos = builder.build(train_sp, train_co,
                                            neutral=train_neutral, thin=True)

        # --- BCE step (GNN + classifier), weighted positives -------------
        encoder.train()
        model.train()
        optimizer.zero_grad()
        sp_emb = encoder(x_train_dict, edge_index_dev)["species"]
        pos_logit, _ = model(with_embedding(x_train_pos, train_sp, sp_emb))
        neg_logit, _ = model(with_embedding(x_neg, neg_sp, sp_emb))
        logits = torch.cat([pos_logit, neg_logit])
        targets = torch.cat([
            torch.full_like(pos_logit, 1.0 - LABEL_SMOOTHING),
            torch.full_like(neg_logit, LABEL_SMOOTHING)])
        weights = torch.cat([w_train, torch.ones_like(neg_logit)])
        loss = (F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none") * weights).sum() / weights.sum()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(main_params, max_norm=5.0)
        optimizer.step()

        # --- gate step (climate gate only) -------------------------------
        gate_opt.zero_grad()
        g_mean = torch.cat([x_train_pos["niche_mean"],
                            x_neg["niche_mean"][-n_easy_now:]])
        g_std = torch.cat([x_train_pos["niche_std"],
                           x_neg["niche_std"][-n_easy_now:]])
        g_tgt = torch.cat([x_train_pos["target_climate"],
                           x_neg["target_climate"][-n_easy_now:]])
        _, dist = InvasionRiskModel.niche_distance(g_mean, g_std, g_tgt)
        target = torch.exp(-dist.squeeze(1).pow(2) / (2 * gate_sigma ** 2))
        gate_loss = F.mse_loss(model.gate(g_mean, g_std, g_tgt).squeeze(1),
                               target)
        gate_loss.backward()
        gate_opt.step()
        gate_sched.step(gate_loss.item())

        # --- validation (scan mode, matched negatives) ---------------------
        encoder.eval()
        model.eval()
        with torch.no_grad():
            sp_emb_v = encoder(x_scan_dict, edge_index_dev)["species"]
            vp, vp_gate = model(with_embedding(x_val_pos, val_sp, sp_emb_v))
            vn, vn_gate = model(with_embedding(x_val_neg, val_neg_sp,
                                               sp_emb_v))
        val_auc = roc_auc_score(y_val, torch.cat([vp, vn]).cpu().numpy(),
                                sample_weight=sw_val)
        gate_sep = float(vp_gate.mean() - vn_gate.mean())
        scheduler.step(val_auc)
        score = val_auc + GATE_SCORE_WEIGHT * max(gate_sep, 0.0)
        history.append({"epoch": epoch, "loss": loss.item(),
                        "gate_loss": gate_loss.item(), "val_auc": val_auc,
                        "gate_sep": gate_sep})
        if score > best["score"] + 1e-3:
            best = {"score": score, "epoch": epoch, "auc": val_auc,
                    "gate_sep": gate_sep,
                    "encoder": copy.deepcopy(encoder.state_dict()),
                    "model": copy.deepcopy(model.state_dict())}
            patience = 0
        else:
            patience += 1
        if verbose and (epoch == 1 or epoch % 25 == 0):
            print(f"   [{label}] epoch {epoch:3d} | BCE {loss.item():.4f} | "
                  f"gate {gate_loss.item():.5f} | val AUC {val_auc:.4f} | "
                  f"gate sep {gate_sep:+.3f} | "
                  f"lr {optimizer.param_groups[0]['lr']:.1e}")
        if patience >= EARLY_STOP_PATIENCE:
            break

    # --- final metrics on the restored best model --------------------------
    encoder.load_state_dict(best["encoder"])
    model.load_state_dict(best["model"])
    encoder.eval()
    model.eval()
    with torch.no_grad():
        sp_emb_f = encoder(x_scan_dict, edge_index_dev)["species"]
        vp, vp_gate = model(with_embedding(x_val_pos, val_sp, sp_emb_f))
        vn, vn_gate = model(with_embedding(x_val_neg, val_neg_sp, sp_emb_f))
    pos_probs = torch.sigmoid(vp).cpu().numpy()
    neg_probs = torch.sigmoid(vn).cpu().numpy()
    roc, pr, fpr, tpr, precision, recall = weighted_metrics(
        pos_probs, neg_probs, w_val_np)

    # uniform negatives (the version-2 definition of the AUC)
    u_sp, u_co = sample_unlabeled(len(val_sp) * VAL_NEG_RATIO, val_pool)
    u_logit, _ = score_pairs(model, builder, u_sp, u_co, sp_emb_f)
    roc_uniform = weighted_metrics(pos_probs,
                                   torch.sigmoid(u_logit).cpu().numpy(),
                                   w_val_np)[0]
    # actual effort features: what documentation alone would add
    roc_raw = roc
    if EFFORT_CORRECTION == "neutralize":
        with torch.no_grad():
            sp_emb_r = encoder(x_dict_raw, edge_index_dev)["species"]
        rp, _ = score_pairs(model, builder, val_sp, val_co, sp_emb_r,
                            neutral=False)
        rn, _ = score_pairs(model, builder, val_neg_sp, val_neg_co,
                            sp_emb_r, neutral=False)
        roc_raw = roc_auc_score(y_val, torch.cat([rp, rn]).cpu().numpy(),
                                sample_weight=sw_val)

    cand_mask = torch.zeros_like(eligible_mask)
    cand_mask[val_pool] = True

    def score_fn(s, c):
        return score_pairs(model, builder, s, c, sp_emb_f)[0]

    rank = (ranking_eval(score_fn, val_sp, val_co, cand_mask)
            if rank_eval else {})
    result = {
        "label": label, "roc_auc": float(roc), "pr_auc": float(pr),
        "roc_auc_uniform_negatives": float(roc_uniform),
        "roc_auc_effort_raw": float(roc_raw),
        "best_epoch": best["epoch"], "gate_sigma": gate_sigma,
        "gate_sep": float(vp_gate.mean() - vn_gate.mean()),
        "rank_metrics": rank, "history": history,
        "encoder": encoder, "model": model, "builder": builder,
        "x_scan_dict": x_scan_dict, "bio_neutral": bio_neutral,
        "val_sp": val_sp, "val_co": val_co,
        "val_neg_sp": val_neg_sp, "val_neg_co": val_neg_co,
        "pos_probs": pos_probs, "neg_probs": neg_probs,
        "pos_gate": vp_gate.cpu().numpy(), "neg_gate": vn_gate.cpu().numpy(),
        "fpr": fpr, "tpr": tpr, "precision": precision, "recall": recall,
    }
    if verbose:
        msg = (f"   [{label}] best epoch {best['epoch']} | ROC-AUC "
               f"{roc:.4f} (uniform negatives {roc_uniform:.4f}, actual "
               f"effort {roc_raw:.4f}) | PR-AUC {pr:.4f} | gate sep "
               f"{result['gate_sep']:+.3f}")
        if rank:
            msg += (f"\n   ranking: median held-out positive in top "
                    f"{rank['rank_median_top_frac']:.1%} | top-5% "
                    f"{rank['rank_hit_top5pct']:.1%} | top-10% "
                    f"{rank['rank_hit_top10pct']:.1%}")
            per_class = [
                f"{lab} {rank[f'rank_hit_top10pct_support_{lab}']:.1%}"
                for lab in SUPPORT_LABELS
                if f"rank_hit_top10pct_support_{lab}" in rank]
            if per_class:
                msg += " | top-10% by support: " + ", ".join(per_class)
        print(msg)
    return result

## Step 11 · Cross-validation

Three complementary tests. Each reports the metrics of Step 10.

| Test | Held out | Question |
|---|---|---|
| **(a) Random pairs**, `N_FOLDS` folds | random 20 % of the positive pairs | Given a species' other records, can we find a missing one? |
| **(b) Species held out**, `N_FOLDS` folds | all positives of 20 % of the species; their negatives come only from held-out species | Does the model work for species without any alien record yet? (Their native and GBIF range stay available, as for a real new candidate.) |
| **(c) Temporal** | positives first recorded in or after `TEMPORAL_CUTOFF_YEAR`; undated and earlier ones train | Could the model have predicted the invasions of the last decades? |

Caveats of (c): GBIF presence, trade and partner presence are not dated,
so they contain some information from after the cutoff; the held-out
pairs themselves are removed from the range data.

**Reading the limited-data columns.** `rank_hit_top10pct_support_low`
vs `…_high` shows whether poorly supported species are found as well as
well-supported ones. `roc_auc_effort_raw` − `roc_auc` is roughly what
documentation alone would add to the AUC: a large gap means the
correction removes a strong (but uninformative for new species) signal.
If the scan-mode metrics also fall far below `roc_auc_effort_raw` in the
species-held-out test, compare with `EFFORT_CORRECTION = "drop"`: it
trains without the effort features, so training and scanning see the
same inputs.

In [ ]:
print("Step 11: cross-validation")
cv_results = {}
METRIC_KEYS = ["roc_auc", "pr_auc", "roc_auc_uniform_negatives",
               "roc_auc_effort_raw"]


def summarise(name, results):
    """Print and return mean/SD of the metrics of a list of splits."""
    summary = {"n_splits": len(results)}
    for k in METRIC_KEYS:
        vals = [r[k] for r in results]
        summary[k] = {"mean": float(np.mean(vals)), "sd": float(np.std(vals))}
    rank_keys = sorted({k for r in results for k in r["rank_metrics"]
                        if not k.startswith("rank_n_")})
    for k in rank_keys:
        vals = [r["rank_metrics"][k] for r in results
                if k in r["rank_metrics"]]
        summary[k] = {"mean": float(np.mean(vals)), "sd": float(np.std(vals))}
    summary["folds"] = [{k: r[k] for k in ["label", "best_epoch", "gate_sep"]
                         + METRIC_KEYS} | r["rank_metrics"] for r in results]
    print(f"\n=== {name} ===")
    for k, v in summary.items():
        if isinstance(v, dict):
            print(f"   {k:38s} {v['mean']:.4f} +/- {v['sd']:.4f}")
    return summary


def light(result):
    """Drop the heavy objects of a split result once it is summarised."""
    for key in ("encoder", "model", "builder", "history", "x_scan_dict",
                "bio_neutral"):
        result.pop(key, None)
    return result


all_idx = torch.arange(n_pos, device=DEVICE)

# ---- (a) random pairs ----------------------------------------------------
if RUN_RANDOM_CV:
    set_seed(0)
    perm = torch.randperm(n_pos, device=DEVICE)
    folds = torch.chunk(perm, N_FOLDS)
    res_a = []
    for f, val_idx in enumerate(folds):
        train_idx = torch.cat([folds[g] for g in range(N_FOLDS) if g != f])
        res_a.append(light(train_and_evaluate_split(
            train_idx, val_idx, f"random{f + 1}")))
    cv_results["random_pairs"] = summarise("random-pair CV", res_a)

# ---- (b) species held out ----------------------------------------------
if RUN_SPECIES_CV:
    gen = torch.Generator().manual_seed(0)
    pos_species = torch.unique(pos_sp_all.cpu())
    shuffled = pos_species[torch.randperm(len(pos_species), generator=gen)]
    fold_of_sp = torch.full((num_species,), -1, dtype=torch.long)
    fold_of_sp[shuffled] = torch.arange(len(shuffled)) % N_FOLDS
    fold_of_sp = fold_of_sp.to(DEVICE)
    pair_fold = fold_of_sp[pos_sp_all]
    res_b = []
    for f in range(N_FOLDS):
        in_fold = fold_of_sp == f
        res_b.append(light(train_and_evaluate_split(
            all_idx[pair_fold != f], all_idx[pair_fold == f],
            f"species{f + 1}",
            train_pool=(eligible_mask & ~in_fold).nonzero(as_tuple=True)[0],
            val_pool=(eligible_mask & in_fold).nonzero(as_tuple=True)[0])))
    cv_results["species_held_out"] = summarise("species-held-out CV", res_b)

# ---- (c) temporal ------------------------------------------------------
if RUN_TEMPORAL_CV:
    late = torch.as_tensor(np.nan_to_num(pos_year, nan=-1)
                           >= TEMPORAL_CUTOFF_YEAR, device=DEVICE)
    n_late = int(late.sum())
    if n_late >= MIN_TEMPORAL_POSITIVES:
        res_c = train_and_evaluate_split(all_idx[~late], all_idx[late],
                                         f"temporal{TEMPORAL_CUTOFF_YEAR}")
        cv_results["temporal"] = summarise(
            f"temporal hold-out (first record >= {TEMPORAL_CUTOFF_YEAR})",
            [light(res_c)])
        cv_results["temporal"]["n_held_out"] = n_late
    else:
        print(f"   temporal hold-out skipped: only {n_late} dated positives "
              f">= {TEMPORAL_CUTOFF_YEAR}")

with open(OUT_DIR / "cv_results.json", "w") as fh:
    json.dump(cv_results, fh, indent=2)
print(f"\n   saved {OUT_DIR / 'cv_results.json'}")

## Step 12 · Production ensemble

`ENSEMBLE_SIZE` models are trained on all positives with different seeds
(different initialisation, negatives, thinning, dropped edges and a
different 10 % early-stopping slice). Their spread is the part of the
uncertainty that MC dropout inside one model misses (Lakshminarayanan
et al. 2017). **Report performance from Step 11**, not from these
early-stopping slices.

All members share one set of effort views, with the typical effort values
taken over **all** positive species, so their scores are comparable and
can be averaged. For scanning, a feature builder over the **full** known
range is used (`scan_builder`). Scan candidates are unrecorded pairs, so
the leave-one-out has no effect on them, and training and scanning see
the same kind of input.

Each member keeps two species embeddings: `sp_emb` from the effort-neutral
node features (used for all scores) and `sp_emb_raw` from the actual
ones (only for the `score_effort_raw` / `effort_shift` columns).

In [ ]:
print("Step 12: production ensemble")
all_pos_species = torch.unique(pos_sp_all)
prod_results = []
for m in range(ENSEMBLE_SIZE):
    seed = 99 + m
    set_seed(seed)
    perm = torch.randperm(n_pos, device=DEVICE)
    n_prod_val = max(1, int(PRODUCTION_VAL_FRACTION * n_pos))
    prod_results.append(train_and_evaluate_split(
        perm[n_prod_val:], perm[:n_prod_val], f"production{m + 1}",
        rank_eval=(m == 0), seed=seed, effort_ref=all_pos_species))
prod = prod_results[0]
x_scan_dict, bio_neutral = prod["x_scan_dict"], prod["bio_neutral"]
scan_builder = PairFeatureBuilder(static, base_range_mat, SRC_STRENGTH_AGGS,
                                  FEATURE_CHUNK, bio_neutral=bio_neutral)
members = []
for res_m in prod_results:
    res_m["encoder"].eval()
    res_m["model"].eval()
    with torch.no_grad():
        members.append({
            "model": res_m["model"], "encoder": res_m["encoder"],
            "sp_emb": res_m["encoder"](x_scan_dict,
                                       edge_index_dev)["species"],
            "sp_emb_raw": res_m["encoder"](x_dict_raw,
                                           edge_index_dev)["species"]})
model, encoder = members[0]["model"], members[0]["encoder"]
final_sp_emb = members[0]["sp_emb"]
GATE_SIGMA = prod["gate_sigma"]
cv_results["production_early_stopping_slice"] = {
    k: prod[k] for k in ["best_epoch", "gate_sep"] + METRIC_KEYS
} | prod["rank_metrics"]
print(f"   ensemble of {len(members)} model(s) ready (best epochs "
      f"{[r['best_epoch'] for r in prod_results]})")

## Step 13 · Diagnostics

1. **Dashboard** of production member 1: ROC, precision–recall, score
   distributions and climate-gate distributions (on its early-stopping
   slice).
2. **Training curves.**
3. **Leakage checks**
   * what the *version-1* sampling rule (exclude only positives, all
     species) would have drawn: share of recorded pairs and of species
     outside the pool;
   * climate distance alone as a classifier, with the leaky niche (target
     inside the range) vs the leave-one-out niche: the leaky AUC should
     be clearly higher, which is the size of the shortcut that was
     removed.
4. **Gate calibration:** gate value vs its Gaussian target along the
   range of niche distances.
5. **Limited-data diagnostics (new).** For `BIAS_DIAG_LOCATIONS`
   locations (the export locations plus random ones) all candidates are
   scored twice: in scan mode and with the actual effort features. Per
   mode, averaged over locations:
   * `spearman_score_effort` / `_support`: rank correlation between the
     score and the effort score / data support of the candidates. Near 0
     (or clearly lower than with actual effort) is what the correction
     aims for. A remaining positive value is partly genuine (range size
     carries "invasive elsewhere") and partly bias that the inputs cannot
     separate.
   * `share_top_<class>` vs `share_candidates_<class>`: how the top
     `BIAS_DIAG_TOP_K` of each list is made up by data-support class,
     compared with all candidates. Strong over-representation of `high`
     means the list still favours well-documented species.

In [ ]:
def plot_dashboard(result, title):
    """ROC, PR, score and gate distributions for one split."""
    fig, ax = plt.subplots(1, 4, figsize=(22, 4.5))
    ax[0].plot(result["fpr"], result["tpr"],
               label=f"AUC {result['roc_auc']:.3f}")
    ax[0].plot([0, 1], [0, 1], "--", color="grey")
    ax[0].set(xlabel="False positive rate", ylabel="True positive rate",
              title="ROC (positives vs matched unlabeled)")
    ax[1].plot(result["recall"], result["precision"],
               label=f"AUC {result['pr_auc']:.3f}")
    ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall")
    for a in ax[:2]:
        a.legend()
    ax[2].hist(result["pos_probs"], bins=40, alpha=0.6, density=True,
               label="held-out aliens")
    ax[2].hist(result["neg_probs"], bins=40, alpha=0.6, density=True,
               label="unlabeled pairs")
    ax[2].set(xlabel="Predicted probability", title="Scores")
    ax[3].hist(result["pos_gate"], bins=30, alpha=0.6, density=True,
               label="held-out aliens")
    ax[3].hist(result["neg_gate"], bins=30, alpha=0.6, density=True,
               label="unlabeled pairs")
    ax[3].set(xlabel="Climate gate", title="Climate gate")
    for a in ax[2:]:
        a.legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"dashboard_{slugify(title)}.png", dpi=130)
    plt.show()


plot_dashboard(prod, "Production model (member 1)")
hist = pd.DataFrame(prod["history"])
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
for a, col in zip(ax, ["loss", "val_auc", "gate_sep"]):
    a.plot(hist["epoch"], hist[col])
    a.set(xlabel="epoch", title=col)
fig.tight_layout()
plt.show()

In [ ]:
print("Leakage and sampling checks")
with torch.no_grad():
    # (1) version-1 negative rule: any species, only positives excluded
    m = 200_000
    sp = torch.randint(num_species, (m,), device=DEVICE)
    co = torch.randint(num_countries, (m,), device=DEVICE)
    pos_only = torch.zeros_like(exclude_mask)
    pos_only[pos_sp_all, pos_co_all] = True
    old = ~pos_only[sp, co]
    print(f"   version-1 rule: recorded pairs among negatives "
          f"{exclude_mask[sp[old], co[old]].float().mean():.1%}, species "
          f"outside the pool {(~eligible_mask[sp[old]]).float().mean():.1%}")
    n_sp, n_co = sample_unlabeled(50_000, eligible_species)
    print(f"   current rule: recorded pairs among negatives "
          f"{exclude_mask[n_sp, n_co].float().mean():.1%} (must be 0)")

    # (2) climate distance alone: leaky vs leave-one-out niche
    vp_sp, vp_co = prod["val_sp"], prod["val_co"]
    vn_sp, vn_co = prod["val_neg_sp"], prod["val_neg_co"]

    def niche_auc(builder, loo):
        """AUC of -distance for held-out positives vs unlabeled pairs."""
        def dist(sp_, co_):
            mean, std = builder.niche(sp_, co_, loo=loo)
            _, d = InvasionRiskModel.niche_distance(
                mean, std, static["co_niche"][co_])
            return d.squeeze(1).cpu().numpy()
        dp, dn = dist(vp_sp, vp_co), dist(vn_sp, vn_co)
        return roc_auc_score(np.r_[np.ones(len(dp)), np.zeros(len(dn))],
                             -np.r_[dp, dn])

    print(f"   climate-distance AUC, leaky niche (target in range): "
          f"{niche_auc(scan_builder, loo=False):.4f}")
    print(f"   climate-distance AUC, leave-one-out niche:          "
          f"{niche_auc(prod['builder'], loo=True):.4f}")

    # (3) gate calibration on real held-out pairs
    sp_all = torch.cat([vp_sp, vn_sp])
    co_all_ = torch.cat([vp_co, vn_co])
    mean, std = prod["builder"].niche(sp_all, co_all_)
    tgt = static["co_niche"][co_all_]
    _, d = InvasionRiskModel.niche_distance(mean, std, tgt)
    g = model.gate(mean, std, tgt).squeeze(1)
    order = torch.argsort(d.squeeze(1))
    print("\n   distance | gate  | target | pair")
    for pos in torch.linspace(0, len(order) - 1, 8).long():
        i = order[pos]
        dv = float(d[i])
        print(f"   {dv:8.2f} | {float(g[i]):.3f} | "
              f"{math.exp(-dv ** 2 / (2 * GATE_SIGMA ** 2)):.3f}  | "
              f"{taxon_names[int(sp_all[i])]} -> "
              f"{all_country_names[int(co_all_[i])]}")

    # (4) niche width by range size: shrinkage should keep small ranges
    #     from getting extreme widths
    n_rng = base_range_mat.sum(dim=1)
    print("\n   niche SD (mean over axes) by range size, candidates:")
    for lo, hi in ((1, 1), (2, 3), (4, 9), (10, 10 ** 6)):
        sel = eligible_species[(n_rng[eligible_species] >= lo)
                               & (n_rng[eligible_species] <= hi)]
        if len(sel) == 0:
            continue
        sel = sel[:2000]
        _, sd = scan_builder.niche(sel, torch.zeros_like(sel), loo=False)
        print(f"   {lo:>3}-{hi if hi < 10 ** 6 else 'max':>3} locations: "
              f"{float(sd.mean()):.3f} ({len(sel):,} species)")

In [ ]:
print("Limited-data diagnostics")
diag_locs = ([] if EXPORT_LOCATIONS == "all"
             else [c for c in EXPORT_LOCATIONS if c in co_to_idx])
rng = np.random.default_rng(SEED)
others = [c for c in all_country_names if c not in diag_locs]
n_more = max(0, min(len(others), BIAS_DIAG_LOCATIONS - len(diag_locs)))
diag_locs += list(rng.choice(others, size=n_more, replace=False))
modes = [("scan", False)]
if EFFORT_CORRECTION == "neutralize":
    modes.append(("actual_effort", True))
diag_rows = []
for loc in diag_locs:
    c = co_to_idx[loc]
    cand = (eligible_mask & ~exclude_mask[:, c]).nonzero(as_tuple=True)[0]
    if len(cand) < BIAS_DIAG_TOP_K:
        continue
    cand_np = cand.cpu().numpy()
    for mode, raw in modes:
        lg = ensemble_logits(members, scan_builder, cand,
                             torch.full_like(cand, c), raw=raw).cpu().numpy()
        top = cand_np[np.argsort(-lg)[:BIAS_DIAG_TOP_K]]
        row = {"location": loc, "mode": mode, "n_candidates": len(cand_np),
               "spearman_score_effort": rank_correlation(
                   lg, effort_score[cand_np]),
               "spearman_score_support": rank_correlation(
                   lg, data_support[cand_np])}
        for j, lab in enumerate(SUPPORT_LABELS):
            row[f"share_candidates_{lab}"] = float(
                (support_class[cand_np] == j).mean())
            row[f"share_top_{lab}"] = float((support_class[top] == j).mean())
        diag_rows.append(row)
bias_diag = pd.DataFrame(diag_rows)
if len(bias_diag):
    bias_summary = bias_diag.groupby("mode").mean(numeric_only=True)
    print(f"   {bias_diag['location'].nunique()} locations, top "
          f"{BIAS_DIAG_TOP_K} per list\n")
    print(bias_summary.T.round(3).to_string())
    cv_results["limited_data_diagnostics"] = {
        "locations": sorted(bias_diag["location"].unique().tolist()),
        "top_k": BIAS_DIAG_TOP_K,
        "by_mode": bias_summary.round(4).to_dict(orient="index")}
    bias_diag.to_csv(OUT_DIR / "limited_data_diagnostics.csv", index=False)
else:
    print("   no location with enough candidates")

# ranking per data-support class from the cross-validation
rows = []
for test, summary in cv_results.items():
    if not isinstance(summary, dict) or test == "limited_data_diagnostics":
        continue
    row = {"test": test}
    for lab in SUPPORT_LABELS:
        v = summary.get(f"rank_hit_top10pct_support_{lab}")
        row[f"top10_{lab}"] = (v["mean"] if isinstance(v, dict) else v)
    rows.append(row)
if rows:
    print("\n   held-out positives in the top 10 % of their location's "
          "list, by data support:")
    print(pd.DataFrame(rows).set_index("test").round(3).to_string())
with open(OUT_DIR / "cv_results.json", "w") as fh:
    json.dump(cv_results, fh, indent=2, default=str)

## Step 14 · Static checkpoint

Everything needed to score, explain and draw saliency maps **without the
data files or retraining** goes into `horizon_scan_static.pt`: per
ensemble member the model and GNN weights, both species embeddings and
the attribution baseline; the static tensors (incl. the shrunk biotic
opportunity), the effort-neutral biotic part, the range and exclusion
masks, both versions of the graph node features, names and settings.

**Attribution baseline**, per member: the mean of every leaf over a
random sample of 2 000 candidate pairs ("an average candidate"), in scan
mode. The contributions in the export are measured against it, so a
positive value means "this feature raises the risk compared with an
average candidate".

Checkpoints of earlier versions cannot be loaded (different inputs).

In [ ]:
print("Step 14: static checkpoint")
set_seed()
with torch.no_grad():
    ref_sp, ref_co = sample_unlabeled(2000, eligible_species)
    ref_leaves = scan_builder.build(ref_sp, ref_co, neutral=True)
    for mem in members:
        ref = with_embedding(ref_leaves, ref_sp, mem["sp_emb"])
        mem["baseline"] = {k: ref[k].mean(dim=0, keepdim=True)
                           for k in LEAF_NAMES}

graph_ids = {"species": np.asarray(all_species_ids, dtype=np.int64)}
for rank in RANKS:
    graph_ids[rank] = np.asarray(sorted(tax_nodes[rank]), dtype=np.int64)

meta = {
    "niche_cols": niche_cols, "context_cols": context_cols,
    "trade_channel_names": trade_channel_names,
    "src_aggs": list(SRC_STRENGTH_AGGS),
    "all_species_ids": all_species_ids,
    "all_country_names": all_country_names,
    "species_info": species_info, "sp_feature_names": sp_feature_names,
    "effort_features": EFFORT_FEATURES,
    "effort_correction": EFFORT_CORRECTION,
    "support_labels": SUPPORT_LABELS,
    "node_types": node_types, "edge_types": edge_types,
    "n_sp_features": n_sp_features, "emb_dim": EMB_DIM,
    "graph_ids": graph_ids, "taxon_node_names": taxon_node_names,
    "gate_sigma": GATE_SIGMA, "ensemble_size": len(members),
    "neighbour_radius_km": NEIGHBOUR_RADIUS_KM,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": {k: v for k, v in globals().items()
               if k.isupper() and isinstance(v, (int, float, str, list,
                                                 tuple, bool))},
}


def cpu(obj):
    """Move tensors in (nested) dicts to the CPU."""
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: cpu(v) for k, v in obj.items()}
    return obj


checkpoint = {
    "format_version": 3,
    "model_dims": model_dims,
    "members": [{"model_state": cpu(m["model"].state_dict()),
                 "encoder_state": cpu(m["encoder"].state_dict()),
                 "sp_emb": cpu(m["sp_emb"]),
                 "sp_emb_raw": cpu(m["sp_emb_raw"]),
                 "ig_baseline": cpu(m["baseline"])} for m in members],
    "static": cpu(static),
    "bio_neutral": cpu(bio_neutral),
    "range_mat": base_range_mat.bool().cpu(),
    "exclude_mask": exclude_mask.cpu(),
    "eligible_mask": eligible_mask.cpu(),
    "x_dict": cpu(x_scan_dict),
    "x_dict_raw": cpu(x_dict_raw),
    "edge_index_dict": cpu(edge_index_dict),
    "meta": meta,
    "metrics": cv_results,
}
torch.save(checkpoint, STATIC_CKPT)
print(f"   saved {STATIC_CKPT} "
      f"({STATIC_CKPT.stat().st_size / 1e6:.0f} MB)")

## Step 15 · Scanner: ranking, uncertainty, contributions, GNN saliency

`Scanner` wraps the trained ensemble (live, or re-loaded in Step 17) and
does four things for one location. Everything is in scan mode
(effort-neutral).

1. **Rank** every candidate (eligible species, not recorded in the
   location) by the ensemble's mean logit (`score_deterministic` =
   sigmoid of it). With `EFFORT_CORRECTION = "neutralize"` the same is
   done with the actual effort features (`score_effort_raw`);
   `effort_shift` = `score_deterministic` − `score_effort_raw` shows how
   much the correction moved a species (negative: the species was pushed
   up by its documentation).
2. **Uncertainty** for the top `EXPORT_TOP_N`: Monte-Carlo dropout in
   every member (`N_MC_SAMPLES` passes each). `probability` is the mean
   over all passes, `uncertainty` their SD (model + ensemble spread),
   `uncertainty_ensemble` the SD between the member means, and
   `uncertainty_se` = SD / √(passes). None of these see missing data;
   that is what `data_support` is for.
3. **Feature contributions** for the top `EXPLAIN_TOP_N`: Integrated
   Gradients on the logit from the average-candidate baseline to the
   pair, along a straight line in `N_IG_STEPS` steps, for every member;
   the attributions are averaged. They add up to the mean logit of the
   pair minus the mean baseline logit; the remaining
   `ig_completeness_gap` should be close to 0 (raise `N_IG_STEPS` if
   not). Each leaf element becomes one contribution with a front-end
   **block**: climate, trade, biotic, relational, spread or location. The
   effort-neutral biotic inputs (has / number of partners) are the same
   for every candidate, so they contribute 0.
4. **GNN saliency map** for the top `SALIENCY_TOP_N`: which graph nodes
   drive the relational-ecology part of the prediction, averaged over the
   members. The gradient of the pair's logit with respect to every node's
   input features is taken through the GNN; per node:
   * `saliency_l2` = ‖∇ₓ logit‖₂ (sensitivity);
   * `contribution` = Σ ∇ₓ logit · x (gradient × input: signed, so
     red/blue like the contribution bars);
   * `contribution_share` = contribution / Σ|contribution| of the map.

   With 2 GNN layers only nodes within **two hops** (along incoming
   edges) can matter. For each kept node the **path** to the focal
   species is reported, and the edges between kept nodes are exported so
   the front-end can draw the local graph. Gradient × input is a
   first-order, local explanation, not an exact decomposition.

In [ ]:
WINDOW_LABELS = {"full": "1995-2024", "recent": "2015-2024"}
AGG_LABELS = {"mean": "mean", "max": "strongest"}
SCAN_INFO_COLUMNS = ["gbif_id", "taxon_name", "group", "kingdom", "phylum",
                     "class", "order", "family", "eicat",
                     "n_alien_locations", "n_native_locations",
                     "data_support", "data_support_class",
                     "n_range_locations", "n_partners", "n_pos_partners",
                     "n_neg_partners", "native_range_known",
                     "gbif_fetch_failed"]


def channel_label(name):
    """'plants_for_planting__recent' -> 'plants for planting (2015-2024)'."""
    base, _, window = name.partition("__")
    return f"{base.replace('_', ' ')} ({WINDOW_LABELS.get(window, window)})"


def leaf_feature_table(meta):
    """One row per explained input element: leaf, index, block, label, key.

    A missing ``index`` (NaN) means the leaf is summed into one
    contribution.
    """
    rows = []

    def add(leaf, items, block):
        for i, (name, key) in enumerate(items):
            rows.append((leaf, i, block, name, key))

    niche = meta["niche_cols"]
    add("target_climate", [(f"target: {pretty_feature(c)}",
                            f"attr_climate_target_{c}") for c in niche],
        "climate")
    add("niche_mean", [(f"species niche (mean): {pretty_feature(c)}",
                        f"attr_climate_niche_mean_{c}") for c in niche],
        "climate")
    add("niche_std", [(f"species niche (spread): {pretty_feature(c)}",
                       f"attr_climate_niche_std_{c}") for c in niche],
        "climate")
    add("koppen", [("Koppen-Geiger profile overlap", "attr_koppen_overlap"),
                   ("Koppen-Geiger class coverage", "attr_koppen_coverage"),
                   ("Koppen-Geiger data available",
                    "attr_koppen_available")], "climate")
    radius = int(meta.get("neighbour_radius_km", 1000))
    add("proximity", [
        ("closeness of nearest established range",
         "attr_proximity_nearest_range"),
        (f"range locations within {radius} km",
         "attr_proximity_n_range_neighbours"),
        ("share of range in the same region",
         "attr_proximity_same_region_share"),
        ("size of the known range", "attr_proximity_range_size")],
        "spread")
    add("context", [(pretty_feature(c), f"attr_context_{c}")
                    for c in meta["context_cols"]], "location")
    chans = meta["trade_channel_names"]
    add("trade_import", [(f"import: {channel_label(c)}",
                          f"attr_trade_import_{c}") for c in chans], "trade")
    add("trade_export", [(f"routed from range: {channel_label(c)}",
                          f"attr_trade_export_{c}") for c in chans], "trade")
    add("trade_source", [
        (f"range exporter strength ({AGG_LABELS.get(a, a)}): "
         f"{channel_label(c)}", f"attr_trade_source_{a}_{c}")
        for a in meta["src_aggs"] for c in chans], "trade")
    add("biotic", [
        ("helpful partners present", "attr_biotic_opp_positive"),
        ("harmful partners present", "attr_biotic_opp_negative"),
        ("has known helpful partners", "attr_biotic_has_positive_partners"),
        ("has known harmful partners", "attr_biotic_has_negative_partners"),
        ("number of helpful partners", "attr_biotic_n_positive_partners"),
        ("number of harmful partners", "attr_biotic_n_negative_partners")],
        "biotic")
    rows.append(("sp_emb", None, "relational", "species graph embedding",
                 "attr_relational_ecology_species_gnn"))
    rows.append(("co_latent", None, "relational",
                 "location latent profile",
                 "attr_relational_ecology_country_latent"))
    return pd.DataFrame(rows, columns=["leaf", "index", "block", "name",
                                       "feature_key"])


class Scanner:
    """Score, explain and map an ensemble (live or re-loaded).

    ``members``: list of dicts with ``model``, ``encoder``, ``sp_emb``
    (scan mode), ``sp_emb_raw`` (actual effort features) and
    ``baseline`` (Integrated-Gradients baseline leaves).
    """

    def __init__(self, members, builder, exclude_mask, eligible_mask,
                 meta, x_dict=None, edge_index_dict=None):
        self.members = members
        for m in members:
            m["model"].eval()
            if m.get("encoder") is not None:
                m["encoder"].eval()
        self.builder = builder
        self.exclude = exclude_mask
        self.eligible = eligible_mask
        self.meta = meta
        self.x_dict = x_dict
        self.edges = edge_index_dict
        self.device = members[0]["sp_emb"].device
        self.co_to_idx = {c: i for i, c in
                          enumerate(meta["all_country_names"])}
        self.info = meta["species_info"].reset_index(drop=True)
        self.features = leaf_feature_table(meta)
        self.correct_effort = meta.get("effort_correction") == "neutralize"
        self._in_csr = None

    # ------------------------------------------------------------ scoring
    def location_index(self, location):
        """Row number of a SInAS location name."""
        if location not in self.co_to_idx:
            raise ValueError(f"Unknown location: {location!r}")
        return self.co_to_idx[location]

    def rank(self, location):
        """All candidates of a location, sorted by the ensemble score."""
        c = self.location_index(location)
        cand = (self.eligible & ~self.exclude[:, c]).nonzero(as_tuple=True)[0]
        if len(cand) == 0:
            raise ValueError(f"No candidates for {location}.")
        co = torch.full_like(cand, c)
        logits = ensemble_logits(self.members, self.builder, cand, co)
        out = pd.DataFrame({
            "sp_idx": cand.cpu().numpy(),
            "score_deterministic": torch.sigmoid(logits).cpu().numpy()})
        if self.correct_effort:
            raw = ensemble_logits(self.members, self.builder, cand, co,
                                  raw=True)
            out["score_effort_raw"] = torch.sigmoid(raw).cpu().numpy()
        else:
            out["score_effort_raw"] = out["score_deterministic"]
        return out.sort_values("score_deterministic", ascending=False,
                               ignore_index=True)

    @torch.no_grad()
    def mc_dropout(self, sp, co, n_samples=N_MC_SAMPLES):
        """MC dropout in every member.

        Returns (mean, total SD, SE, SD between member means) of the
        probability, as numpy arrays.
        """
        leaves = self.builder.build(sp, co, neutral=True)
        per_member = []
        for m in self.members:
            model = m["model"]
            model.eval()
            dropouts = [d for d in model.modules()
                        if isinstance(d, nn.Dropout)]
            for d in dropouts:
                d.train()
            x = with_embedding(leaves, sp, m["sp_emb"])
            per_member.append(torch.stack(
                [torch.sigmoid(model(x)[0]) for _ in range(n_samples)]))
            for d in dropouts:
                d.eval()
        probs = torch.stack(per_member)  # (members, passes, pairs)
        flat = probs.reshape(-1, probs.shape[-1])
        sd = flat.std(dim=0)
        between = (probs.mean(dim=1).std(dim=0) if len(self.members) > 1
                   else torch.zeros_like(sd))
        return (flat.mean(dim=0).cpu().numpy(), sd.cpu().numpy(),
                (sd / math.sqrt(flat.shape[0])).cpu().numpy(),
                between.cpu().numpy())

    # ------------------------------------------------- feature contributions
    def explain(self, sp_i, co_i, n_steps=N_IG_STEPS):
        """Integrated Gradients averaged over the members.

        Returns (DataFrame, summary dict). The averaged contributions add
        up to mean logit(pair) - mean logit(baseline) over the members.
        """
        sp = torch.tensor([sp_i], device=self.device)
        co = torch.tensor([co_i], device=self.device)
        leaves = self.builder.build(sp, co, neutral=True)
        alphas = torch.linspace(1.0 / n_steps, 1.0, n_steps,
                                device=self.device).unsqueeze(1)
        attr = {k: 0.0 for k in LEAF_NAMES}
        logit_sum, base_sum = 0.0, 0.0
        for m in self.members:
            model, base = m["model"], m["baseline"]
            model.eval()
            actual = with_embedding(leaves, sp, m["sp_emb"])
            path = {k: (base[k] + alphas * (actual[k] - base[k])
                        ).requires_grad_(True) for k in LEAF_NAMES}
            logits, _ = model(path, detach_gate=False)
            grads = torch.autograd.grad(logits.sum(),
                                        [path[k] for k in LEAF_NAMES],
                                        allow_unused=True)
            for k, g in zip(LEAF_NAMES, grads):
                g = torch.zeros_like(path[k]) if g is None else g
                attr[k] = attr[k] + (g.mean(dim=0)
                                     * (actual[k][0] - base[k][0])).detach()
            with torch.no_grad():
                logit_sum += float(model(actual, detach_gate=False)[0])
                base_sum += float(model(base, detach_gate=False)[0])
        n = len(self.members)
        attr = {k: (v / n).cpu().numpy() for k, v in attr.items()}
        logit, base_logit = logit_sum / n, base_sum / n
        values = [float(attr[leaf].sum()) if pd.isna(idx)
                  else float(attr[leaf][int(idx)])
                  for leaf, idx in zip(self.features["leaf"],
                                       self.features["index"])]
        table = self.features.assign(value=values)
        summary = {"probability": 1 / (1 + math.exp(-logit)),
                   "baseline_probability": 1 / (1 + math.exp(-base_logit)),
                   "ig_completeness_gap": (logit - base_logit) - sum(values)}
        return table, summary

    # ------------------------------------------------------- GNN saliency
    def _in_neighbours(self, node_type, idx):
        """[(src_type, src_idx, relation)] of edges ending in a node."""
        if self._in_csr is None:
            self._in_csr = {}
            for et, ei in self.edges.items():
                dst = ei[1].cpu().numpy()
                order = np.argsort(dst, kind="stable")
                self._in_csr[et] = (dst[order], ei[0].cpu().numpy()[order])
        out = []
        for (st, rel, dt), (dst, src) in self._in_csr.items():
            if dt != node_type:
                continue
            lo, hi = np.searchsorted(dst, [idx, idx + 1])
            out += [(st, int(s), rel) for s in src[lo:hi]]
        return out

    def node_name(self, node_type, idx):
        """Readable name of a graph node."""
        gid = int(self.meta["graph_ids"][node_type][idx])
        if node_type == "species":
            return str(self.info.at[idx, "taxon_name"])
        name = self.meta["taxon_node_names"].get(gid)
        return f"{node_type} {name}" if name else f"{node_type} {gid}"

    def gnn_saliency(self, location, sp_list, top_k=SALIENCY_TOP_K):
        """Node saliency maps for several focal species of one location.

        Gradient x input through each member's GNN, averaged over the
        members. Returns (nodes DataFrame, edges DataFrame).
        """
        if self.x_dict is None or any(m.get("encoder") is None
                                      for m in self.members):
            raise RuntimeError("This scanner has no GNN (graph not saved).")
        c = self.location_index(location)
        leaves = {k: v.detach().clone().requires_grad_(True)
                  for k, v in self.x_dict.items()}
        types = list(leaves)
        sp = torch.as_tensor(sp_list, device=self.device)
        co = torch.full_like(sp, c)
        pair_leaves = self.builder.build(sp, co, neutral=True)
        acc = [dict() for _ in sp_list]  # per focal: (type, idx) -> sums
        for m in self.members:
            emb = m["encoder"](leaves, self.edges)["species"]
            logits, _ = m["model"](with_embedding(pair_leaves, sp, emb))
            for j in range(len(sp_list)):
                grads = torch.autograd.grad(
                    logits[j], [leaves[t] for t in types],
                    retain_graph=True, allow_unused=True)
                for t, g in zip(types, grads):
                    if g is None:
                        continue
                    l2 = g.norm(dim=1)
                    contrib = (g * leaves[t]).sum(dim=1)
                    nz = (l2 > 0).nonzero(as_tuple=True)[0]
                    for i, cv, lv in zip(nz.tolist(), contrib[nz].tolist(),
                                         l2[nz].tolist()):
                        entry = acc[j].setdefault((t, i), [0.0, 0.0])
                        entry[0] += cv
                        entry[1] += lv
            del emb, logits
        n_mem = len(self.members)
        node_rows, edge_rows = [], []
        for j, focal in enumerate(sp.tolist()):
            cands = [(t, i, v[0] / n_mem, v[1] / n_mem)
                     for (t, i), v in acc[j].items()]
            paths = self._paths(focal)
            total = sum(abs(cv) for _, _, cv, _ in cands) or 1.0
            cands.sort(key=lambda r: -abs(r[2]))
            keep = {("species", focal)}
            keep.update((t, i) for t, i, _, _ in cands[:top_k])
            for t, i, _, _ in cands[:top_k]:  # keep connectors of hop 2
                via = paths.get((t, i), {}).get("via")
                if via is not None:
                    keep.add(via)
            focal_id = int(self.meta["graph_ids"]["species"][focal])
            for rank_i, (t, i, cv, l2) in enumerate(cands, start=1):
                if (t, i) not in keep:
                    continue
                p = paths.get((t, i), {"hop": None, "path": ""})
                if (t, i) == ("species", focal):
                    p = {"hop": 0, "path": "focal species (own traits)"}
                node_rows.append({
                    "target_country": location, "gbif_id": focal_id,
                    "taxon_name": self.node_name("species", focal),
                    "node_type": t,
                    "node_gbif_id": int(self.meta["graph_ids"][t][i]),
                    "node_name": self.node_name(t, i),
                    "hop": p["hop"], "relation_path": p["path"],
                    "saliency_l2": l2, "contribution": cv,
                    "contribution_share": cv / total,
                    "saliency_rank": rank_i,
                    "in_top_k": rank_i <= top_k})
            for (t, i), p in paths.items():
                if (t, i) in keep and p.get("edge"):
                    (dt, di), rel = p["edge"]
                    if (dt, di) in keep:
                        edge_rows.append({
                            "target_country": location, "gbif_id": focal_id,
                            "src_type": t,
                            "src_gbif_id": int(self.meta["graph_ids"][t][i]),
                            "dst_type": dt,
                            "dst_gbif_id": int(self.meta["graph_ids"][dt][di]),
                            "relation": rel})
        return pd.DataFrame(node_rows), pd.DataFrame(edge_rows)

    def _paths(self, focal):
        """Shortest incoming path (<= 2 hops) from every node to ``focal``."""
        target = ("species", focal)
        name = self.node_name
        paths = {}
        for st, si, rel in self._in_neighbours("species", focal):
            if (st, si) != target and (st, si) not in paths:
                paths[(st, si)] = {
                    "hop": 1, "via": None, "edge": (target, rel),
                    "path": f"{name(st, si)} -{rel}-> focal"}
        for (mt, mi), p1 in list(paths.items()):
            rel1 = p1["edge"][1]
            for st, si, rel2 in self._in_neighbours(mt, mi):
                if (st, si) == target or (st, si) in paths:
                    continue
                paths[(st, si)] = {
                    "hop": 2, "via": (mt, mi), "edge": ((mt, mi), rel2),
                    "path": (f"{name(st, si)} -{rel2}-> {name(mt, mi)} "
                             f"-{rel1}-> focal")}
        return paths

    # ----------------------------------------------------------- full scan
    def scan(self, location, top_n=EXPORT_TOP_N, explain_n=EXPLAIN_TOP_N,
             saliency_n=SALIENCY_TOP_N, gbif_code=None):
        """Ranked list, contributions and saliency maps for one location.

        Returns a dict of DataFrames: scan, contributions, saliency_nodes,
        saliency_edges.
        """
        c = self.location_index(location)
        ranked = self.rank(location)
        if gbif_code:
            ranked = verify_absent_in_gbif(ranked.head(top_n * 5), self.info,
                                           gbif_code, top_n)
        ranked = ranked.head(top_n).reset_index(drop=True)
        sp = torch.as_tensor(ranked["sp_idx"].values, device=self.device)
        co = torch.full_like(sp, c)
        mean, sd, se, between = self.mc_dropout(sp, co)
        described = {k: v.cpu().numpy() for k, v in
                     self.builder.describe(sp, co).items()}
        scan = pd.DataFrame({
            "sp_idx": ranked["sp_idx"].values,
            "probability": mean, "uncertainty": sd, "uncertainty_se": se,
            "uncertainty_ensemble": between,
            "score_deterministic": ranked["score_deterministic"].values,
            "score_effort_raw": ranked["score_effort_raw"].values,
            **described})
        scan["effort_shift"] = (scan["score_deterministic"]
                                - scan["score_effort_raw"])
        if "gbif_occurrence_count" in ranked.columns:
            scan["gbif_occurrence_count"] = (
                ranked["gbif_occurrence_count"].values)
        scan = scan.sort_values("probability", ascending=False,
                                ignore_index=True)
        scan.insert(0, "rank", np.arange(1, len(scan) + 1))
        info_cols = [col for col in SCAN_INFO_COLUMNS
                     if col in self.info.columns]
        scan = pd.concat([scan[["rank"]],
                          self.info.loc[scan["sp_idx"], info_cols]
                          .reset_index(drop=True),
                          scan.drop(columns="rank")], axis=1)
        scan.insert(3, "target_country", location)

        contrib_frames, extra = [], []
        for sp_i in tqdm(scan["sp_idx"].head(explain_n), desc=f"IG {location}",
                         leave=False):
            table, summary = self.explain(int(sp_i), c)
            gid = int(self.meta["graph_ids"]["species"][sp_i])
            contrib_frames.append(table.drop(columns=["leaf", "index"])
                                  .assign(gbif_id=gid, **summary))
            block_sum = table.groupby("block")["value"].sum()
            top = table.loc[table["value"].abs().nlargest(len(table)).index]
            extra.append({
                "sp_idx": sp_i,
                "top_driver": next((n for n, v in zip(top["name"],
                                                      top["value"]) if v > 0),
                                   ""),
                "top_suppressor": next((n for n, v in zip(top["name"],
                                                          top["value"])
                                        if v < 0), ""),
                "baseline_probability": summary["baseline_probability"],
                "ig_completeness_gap": summary["ig_completeness_gap"],
                **{f"block_contribution_{b}": v
                   for b, v in block_sum.items()}})
        if extra:
            scan = scan.merge(pd.DataFrame(extra), on="sp_idx", how="left")
        scan["has_contributions"] = scan.index < explain_n
        contributions = (pd.concat(contrib_frames, ignore_index=True)
                         if contrib_frames else pd.DataFrame())
        if len(contributions):
            contributions.insert(0, "target_country", location)
            contributions = contributions[
                ["target_country", "gbif_id", "block", "name", "value",
                 "feature_key", "probability", "baseline_probability",
                 "ig_completeness_gap"]]

        nodes, edges = pd.DataFrame(), pd.DataFrame()
        has_gnn = self.x_dict is not None and all(
            m.get("encoder") is not None for m in self.members)
        focal = (scan["sp_idx"].head(saliency_n).astype(int).tolist()
                 if has_gnn else [])
        if focal:
            parts = [self.gnn_saliency(location, focal[i:i + 5])
                     for i in range(0, len(focal), 5)]
            nodes = pd.concat([p[0] for p in parts], ignore_index=True)
            edges = pd.concat([p[1] for p in parts], ignore_index=True)
        scan["has_saliency_map"] = scan.index < len(focal)
        return {"scan": scan.drop(columns="sp_idx"),
                "contributions": contributions,
                "saliency_nodes": nodes, "saliency_edges": edges}


def verify_absent_in_gbif(ranked, info, gbif_code, top_n):
    """Keep the first ``top_n`` candidates with zero live GBIF records."""
    from pygbif import occurrences  # optional dependency

    kept, present, errors = [], 0, 0
    for row in tqdm(ranked.itertuples(index=False), total=len(ranked),
                    desc=f"GBIF {gbif_code}", leave=False):
        if len(kept) >= top_n:
            break
        gid = int(info.at[int(row.sp_idx), "gbif_id"])
        try:
            count = occurrences.search(taxonKey=gid, country=gbif_code,
                                       limit=0).get("count", 0)
        except Exception:  # network error: keep the row, flag it
            count, errors = -1, errors + 1
        if count > 0:
            present += 1
        else:
            kept.append({**row._asdict(), "gbif_occurrence_count": count})
        time.sleep(GBIF_SLEEP_S)
    print(f"   GBIF {gbif_code}: kept {len(kept)}, already present "
          f"{present}, errors {errors}")
    return pd.DataFrame(kept)


def plot_saliency_map(nodes, edges, gbif_id, ax=None):
    """Radial saliency map: focal centre, hop 1 inner ring, hop 2 outer."""
    sub = nodes[nodes["gbif_id"] == gbif_id]
    e = edges[edges["gbif_id"] == gbif_id] if len(edges) else edges
    if sub.empty:
        print("No saliency map for this species.")
        return
    ax = ax or plt.subplots(figsize=(9, 8))[1]
    pos = {}
    for hop, radius in ((1, 1.0), (2, 2.0)):
        ring = sub[sub["hop"] == hop]
        for k, row in enumerate(ring.itertuples()):
            angle = 2 * math.pi * k / max(len(ring), 1)
            pos[(row.node_type, row.node_gbif_id)] = (radius * math.cos(angle),
                                                      radius * math.sin(angle))
    focal = sub[sub["hop"] == 0]
    for row in focal.itertuples():
        pos[(row.node_type, row.node_gbif_id)] = (0.0, 0.0)
    for row in e.itertuples():
        a = pos.get((row.src_type, row.src_gbif_id))
        b = pos.get((row.dst_type, row.dst_gbif_id))
        if a and b:
            ax.plot([a[0], b[0]], [a[1], b[1]], color="grey", alpha=0.4,
                    lw=1)
    vmax = max(sub["contribution"].abs().max(), 1e-12)
    for row in sub.itertuples():
        xy = pos.get((row.node_type, row.node_gbif_id))
        if xy is None:
            continue
        ax.scatter(*xy, s=200 + 1600 * abs(row.contribution) / vmax,
                   c=[row.contribution], cmap="RdBu_r", vmin=-vmax,
                   vmax=vmax, edgecolors="black",
                   linewidths=3 if row.hop == 0 else 1, zorder=3)
        ax.annotate(row.node_name, xy, fontsize=8, ha="center",
                    va="bottom", xytext=(0, 12), textcoords="offset points")
    ax.set_title(f"GNN saliency: {sub['taxon_name'].iloc[0]} -> "
                 f"{sub['target_country'].iloc[0]}\nred raises, blue lowers "
                 "the risk (gradient x input); size = |contribution|")
    ax.axis("off")
    ax.set_aspect("equal")
    plt.show()

## Step 16 · Export for the front-end

For every location in `EXPORT_LOCATIONS` four CSV files are written to
`model_outputs/exports/` (`<loc>` = lower-case location name with
underscores):

| File | One row per | Front-end use |
|---|---|---|
| `scan_<loc>.csv` | candidate species (top `EXPORT_TOP_N`) | list, filters, charts (`probability`, `uncertainty`, `group`, `eicat`) |
| `contributions_<loc>.csv` | species × feature (top `EXPLAIN_TOP_N`) | detail panel bars: `block`, `name`, `value` (logit units), plus `probability` and `baseline_probability` |
| `gnn_saliency_nodes_<loc>.csv` | species × graph node (top `SALIENCY_TOP_N`) | local saliency map |
| `gnn_saliency_edges_<loc>.csv` | species × edge between kept nodes | lines of the map |

**New columns in `scan_<loc>.csv`** (the front-end keeps reading the
files as before; it can use these later): `data_support`,
`data_support_class`, `n_range_locations`, `n_partners`,
`n_pos_partners`, `n_neg_partners`, `native_range_known`,
`gbif_fetch_failed`, `uncertainty_ensemble`, `score_effort_raw`,
`effort_shift`. A species with high `probability` but `data_support_class
= low` rests on little evidence: treat it as "check", not as "confident".

With more than one location the same tables are also combined into
`*_all_locations.csv`. `model_card.json` summarises data, streams,
limited-data controls, settings and metrics for the "About the model"
tab, and `feature_dictionary.csv` lists every contribution with its
block.

In [ ]:
print("Step 16: front-end exports")
scanner = Scanner(members, scan_builder, exclude_mask, eligible_mask, meta,
                  x_dict=x_scan_dict, edge_index_dict=edge_index_dev)
locations = (all_country_names if EXPORT_LOCATIONS == "all"
             else list(EXPORT_LOCATIONS))
combined = defaultdict(list)
for loc in tqdm(locations, desc="Locations"):
    code_ = GBIF_COUNTRY_CODES.get(loc) if GBIF_VERIFY else None
    out = scanner.scan(loc, gbif_code=code_)
    for kind, frame in out.items():
        name = {"scan": "scan", "contributions": "contributions",
                "saliency_nodes": "gnn_saliency_nodes",
                "saliency_edges": "gnn_saliency_edges"}[kind]
        frame.to_csv(EXPORT_DIR / f"{name}_{slugify(loc)}.csv", index=False)
        combined[name].append(frame)
if len(locations) > 1:
    for name, frames in combined.items():
        pd.concat(frames, ignore_index=True).to_csv(
            EXPORT_DIR / f"{name}_all_locations.csv", index=False)

scanner.features.drop(columns="index").to_csv(
    EXPORT_DIR / "feature_dictionary.csv", index=False)


def file_info(path):
    """Name and modification date of an input file."""
    stamp = datetime.fromtimestamp(path.stat().st_mtime, timezone.utc)
    return {"file": path.name, "modified": stamp.date().isoformat()}


model_card = {
    "trained_at": meta["trained_at"],
    "inputs": {k: file_info(p) for k, p in INPUTS.items()},
    "n_species_in_graph": num_species,
    "n_candidate_species": int(eligible_mask.sum()),
    "n_locations": num_countries,
    "n_positive_pairs": n_pos,
    "species_pool": SPECIES_POOL,
    "ensemble_size": len(members),
    "streams": {
        "relational": f"2-layer heterogeneous GraphSAGE ({len(edge_types)} "
                      f"edge types, {n_sp_features} species features, "
                      f"DropEdge {EDGE_DROPOUT}) x location autoencoder "
                      f"({n_co_features} -> {CO_LATENT_DIM})",
        "climate": f"{n_niche} niche variables (shrunk niche width) + "
                   "Koppen-Geiger overlap",
        "trade": f"{num_channels} channels "
                 f"({len(base_channels)} pathways x {list(TRADE_WINDOWS)})",
        "biotic": "helpful / harmful partner presence (shrunk) and "
                  "partner counts",
        "location": f"{n_context} geography and socio-economic features",
        "spread": "nearest range, neighbours within "
                  f"{NEIGHBOUR_RADIUS_KM:.0f} km, same-region share, "
                  "range size",
    },
    "limited_data_controls": {
        "effort_correction": EFFORT_CORRECTION,
        "effort_features": EFFORT_FEATURES,
        "negatives_matched_on": list(MATCH_NEGATIVES_ON),
        "negatives_location_matched": NEG_LOCATION_MATCHED,
        "positive_weight_power": POSITIVE_WEIGHT_POWER,
        "niche_prior_strength": NICHE_PRIOR_STRENGTH,
        "biotic_prior_strength": BIOTIC_PRIOR_STRENGTH,
        "composition_prior_strength": COMPOSITION_PRIOR_STRENGTH,
        "range_thinning": [RANGE_THINNING_PROB, RANGE_THINNING_KEEP_MIN],
        "edge_dropout": EDGE_DROPOUT,
        "data_support_cuts": list(SUPPORT_CLASS_CUTS),
    },
    "uncertainty": f"{len(members)}-model seed ensemble x MC dropout, "
                   f"{N_MC_SAMPLES} passes per model",
    "contributions": f"Integrated Gradients, {N_IG_STEPS} steps, averaged "
                     "over the ensemble; baseline = mean of 2000 random "
                     "candidate pairs",
    "gnn_saliency": f"gradient x input on node features, top "
                    f"{SALIENCY_TOP_K} nodes within 2 hops, averaged over "
                    "the ensemble",
    "metrics": cv_results,
}
with open(EXPORT_DIR / "model_card.json", "w") as fh:
    json.dump(model_card, fh, indent=2, default=str)
print(f"   wrote exports for {len(locations)} location(s) to {EXPORT_DIR}")
example = out["scan"]
display_cols = ["rank", "taxon_name", "group", "probability", "uncertainty",
                "data_support_class", "effort_shift", "top_driver"]
example[[c for c in display_cols if c in example.columns]].head(15)

In [ ]:
# Saliency map of the top-ranked species of the last exported location.
if len(out["saliency_nodes"]):
    top_id = int(out["scan"]["gbif_id"].iloc[0])
    plot_saliency_map(out["saliency_nodes"], out["saliency_edges"], top_id)
    print(out["saliency_nodes"].query("gbif_id == @top_id and in_top_k")
          [["node_name", "node_type", "hop", "relation_path",
            "contribution", "contribution_share"]].to_string(index=False))

## Step 17 · Re-load the checkpoint in a fresh session

To rescore or export later without the data files: run Step 0 (imports,
configuration, helpers), Step 7 (architecture) and Step 15 (scanner),
then the cell below. Everything else comes from the checkpoint.

In [ ]:
def load_static_scanner(path=STATIC_CKPT, device=DEVICE):
    """Rebuild a Scanner from ``horizon_scan_static.pt``."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    check(ckpt.get("format_version") == 3,
          "Checkpoint was written by an older notebook; retrain.")
    meta_ = ckpt["meta"]
    static_ = {k: (v.to(device) if torch.is_tensor(v) else v)
               for k, v in ckpt["static"].items()}
    builder_ = PairFeatureBuilder(
        static_, ckpt["range_mat"].float().to(device), meta_["src_aggs"],
        FEATURE_CHUNK, bio_neutral=ckpt["bio_neutral"].to(device))
    members_ = []
    for mem in ckpt["members"]:
        model_ = InvasionRiskModel(ckpt["model_dims"])
        model_.load_state_dict(mem["model_state"])
        encoder_ = RelationalEcologicalGNN(
            meta_["emb_dim"], meta_["edge_types"], meta_["node_types"],
            meta_["n_sp_features"])
        encoder_.load_state_dict(mem["encoder_state"])
        members_.append({
            "model": model_.to(device), "encoder": encoder_.to(device),
            "sp_emb": mem["sp_emb"].to(device),
            "sp_emb_raw": mem["sp_emb_raw"].to(device),
            "baseline": {k: v.to(device)
                         for k, v in mem["ig_baseline"].items()}})
    return Scanner(
        members_, builder_, ckpt["exclude_mask"].to(device),
        ckpt["eligible_mask"].to(device), meta_,
        x_dict={k: v.to(device) for k, v in ckpt["x_dict"].items()},
        edge_index_dict={k: v.to(device)
                         for k, v in ckpt["edge_index_dict"].items()})


static_scanner = load_static_scanner()
check_loc = (static_scanner.meta["all_country_names"][0]
             if EXPORT_LOCATIONS == "all" else EXPORT_LOCATIONS[0])
reloaded = static_scanner.rank(check_loc).head(5)
reloaded["taxon_name"] = static_scanner.info.loc[reloaded["sp_idx"],
                                                 "taxon_name"].values
print(f"Top 5 for {check_loc} from the re-loaded checkpoint "
      "(should match Step 16):")
reloaded